# PutStrike iTransformer — Per-Stock Model Training

**iTransformer: Inverted Transformers Are Effective for Time Series Forecasting** (ICLR 2024, Liu et al.)

## Overview

Train **individual per-stock iTransformer models** for each of the 80+ screener stocks,
then export to ONNX and push to HuggingFace Hub for client-side inference on the PutStrike website.

### Why Per-Stock Models (Not Universal)

- Each stock has unique volatility characteristics, sector dynamics, and price patterns
- Per-stock models learn stock-specific feature interactions via cross-variate attention
- Eliminates cross-stock contamination — a bank's patterns don't dilute a tech stock's model
- ~2,500 samples per stock is sufficient for the iTransformer architecture (128-dim, 3 layers)
- Individual models allow targeted retraining when a stock's regime changes

### Architecture

- **iTransformer**: each **feature** is a token (cross-variate attention captures how features interact)
- **RevIN**: Reversible Instance Normalization for non-stationary financial time series
- **126 features**: OHLCV technicals + macro + relative strength + advanced volume + regime detection + sector ETF + credit market + industry commodities
- **60-day lookback → 60 trading day forecast** (covers all DTE presets: 7d to 120d)

### Training Design

- **Per-stock training**: one model per stock, trained on that stock's data only
- **Walk-forward validation**: 70/15/15 chronological split (no look-ahead bias)
- **HuberLoss(delta=0.02)**: robust to earnings/event return outliers
- **LR warmup + cosine decay**: standard for Transformers
- **10 years of daily data** per stock via Yahoo Finance

### Feature Categories (126 features)

| Category | Count | Description |
|----------|-------|-------------|
| Price Action | 10 | SMA/EMA crosses, Bollinger, ATR, Keltner |
| Momentum | 15 | RSI, MACD, Stochastic, Williams %R, CCI, Aroon, ROC |
| Volume | 9 | OBV, CMF, relative volume, MFI, A/D line, force index |
| Volatility | 4 | HV 5/10/20/60d |
| Statistical | 14 | Z-scores, percentile ranks, autocorrelation, Hurst exponent |
| Regime | 7 | Vol regime, Parkinson/Garman-Klass vol, tail ratio, consistency |
| Price Structure | 5 | Range position, ATR ratio, consecutive days, candle body |
| Relative Strength | 4 | Returns vs SPY, rolling correlation with market |
| Cross-Asset | 3 | Rolling correlation with VIX, beta to SPY, volume-price corr |
| Intermarket | 4 | SPY momentum, gold/oil ratio, DXY-VIX interaction |
| Macro | 10 | VIX term structure, Treasury yields, USD index, Gold, Oil |
| Calendar | 5 | Day of week, month cycle, OPEX week, quarter end |
| Returns | 5 | 1/5/10/20/60d log returns |
| Drawdown/Gap | 4 | Max drawdown, gap features |
| Trend | 9 | Price slopes, Ichimoku, up/down ratios, higher moments |

### ONNX Export

- Uses the **legacy TorchScript ONNX exporter** (`dynamo=False`) for compatibility
- The dynamo-based exporter (`torch.export.export`) fails on RevIN's dynamic buffer reassignment and string `mode` parameter — these are fundamental to the architecture
- `dynamic_axes` used for batch dimension flexibility (the deprecation warning is suppressed and harmless)
- `onnxscript` is installed as a dependency (required by PyTorch's ONNX infrastructure)
- `enable_nested_tensor=False` on `TransformerEncoder` suppresses the `norm_first` warning
- Per-stock ONNX models (~1.5-2 MB each) with opset 18

### Output

- Per-stock ONNX models (~1.5-2 MB each) pushed to HuggingFace Hub
- Config JSON with feature names, per-stock metrics, architecture details
- Website loads the correct per-stock ONNX model for each ticker

### Setup

1. Run in Google Colab with **L4 GPU** runtime (Runtime > Change runtime type > L4 GPU)
2. Add `HF_TOKEN` and `HF_REPO_ID` as Colab Secrets (key icon in left sidebar)
3. Execute all cells in order (~45-90 min on L4 GPU for all stocks)
4. Models auto-export to ONNX and push to HuggingFace Hub

In [ ]:
# Cell 1: Colab Secrets — Must run FIRST
# Go to the Secrets panel (key icon in the left sidebar) and add:
#   HF_TOKEN      — your HuggingFace write token (https://huggingface.co/settings/tokens)
#   HF_REPO_ID    — e.g. "jcl347/putstrike"
#   FRED_API_KEY  — your FRED API key (https://fred.stlouisfed.org/docs/api/api_key.html)

from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
HF_REPO_ID = userdata.get('HF_REPO_ID')
FRED_API_KEY = userdata.get('FRED_API_KEY')

print(f"[OK] HF_TOKEN loaded (length {len(HF_TOKEN)})")
print(f"[OK] HF_REPO_ID = {HF_REPO_ID}")
print(f"[OK] FRED_API_KEY loaded (length {len(FRED_API_KEY)})")


In [ ]:
# Cell 2: Install Dependencies
import subprocess
import sys

packages = [
    "torch", "numpy", "pandas", "yfinance",
    "scikit-learn", "matplotlib", "onnx", "onnxruntime",
    "onnxscript",  # Required by PyTorch dynamo-based ONNX exporter
    "huggingface_hub",
    "fredapi",  # FRED (Federal Reserve Economic Data) API client
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("[OK] All dependencies installed.")


In [ ]:
# Cell 3: Configuration
import os
import torch
import numpy as np

# ── All 80+ screener stocks ──
SCREENER_SYMBOLS = [
    # Mega-cap tech
    "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA", "AVGO", "ORCL", "CRM",
    "AMD", "INTC", "QCOM", "ADBE", "NFLX", "CSCO", "IBM", "TXN", "NOW", "AMAT",
    "MU", "LRCX", "KLAC", "SNPS", "CDNS", "PANW", "CRWD", "FTNT",
    # Finance
    "JPM", "V", "MA", "BAC", "WFC", "GS", "MS", "AXP", "BLK", "SCHW", "C",
    # Healthcare
    "JNJ", "UNH", "LLY", "PFE", "ABBV", "MRK", "TMO", "ABT", "DHR", "BMY", "AMGN",
    # Consumer
    "PG", "KO", "PEP", "COST", "WMT", "MCD", "NKE", "SBUX", "TGT", "HD", "LOW",
    # Energy
    "XOM", "CVX", "COP", "SLB", "EOG",
    # Industrial
    "CAT", "DE", "HON", "UNP", "RTX", "BA", "GE", "LMT", "MMM",
    # ETFs
    "SPY", "QQQ", "IWM", "DIA", "XLF", "XLE", "XLK", "XLV", "XBI",
    # Other
    "DIS", "PYPL", "SQ", "COIN", "ABNB", "UBER",
]

# ── Macro tickers (additional features) ──
# SPY is included for relative strength features (stock vs market returns)
MACRO_TICKERS = ["^VIX", "^VIX3M", "^TNX", "DX-Y.NYB", "GC=F", "CL=F", "SPY"]

# ── Additional data tickers for v6.0 features ──
CREDIT_TICKERS = ["HYG", "TLT"]      # Credit market ETFs
EXTRA_MACRO_TICKERS = ["^VIX9D", "HG=F", "BTC-USD"]  # VIX9D, copper, bitcoin
SECTOR_ETFS = ["XLK", "XLF", "XLV", "XLE", "XLI", "XLY", "XLP", "XLC", "XLB", "XLU", "XLRE"]
# (ALL_DATA_TICKERS is defined after STOCK_SPECIFIC_DRIVERS below)
# ── FRED (Federal Reserve Economic Data) series for macro features ──
# These are national US economic indicators fetched via the FRED API.
# Requires FRED_API_KEY in Colab Secrets.
FRED_SERIES = {
    "BAMLH0A0HYM2": "fred_hy_spread",          # ICE BofA US High Yield OAS (credit spread)
    "T10Y2Y": "fred_yield_curve",               # 10Y minus 2Y Treasury yield curve
    "T10YIE": "fred_breakeven_inflation",       # 10Y Breakeven Inflation Rate
    "DGS2": "fred_2y_yield",                    # 2-Year Treasury Constant Maturity Rate
    "ICSA": "fred_jobless_claims",              # Initial Jobless Claims (weekly)
    "UMCSENT": "fred_consumer_sentiment",       # UMich Consumer Sentiment (monthly)
    "STLFSI4": "fred_financial_stress",         # St. Louis Fed Financial Stress Index (weekly, replaced STLFSI2)
    "T10Y3M": "fred_t10y3m_spread",            # 10Y minus 3M Treasury spread (recession signal)
    "DFF": "fed_funds_rate",                    # Daily Federal Funds Effective Rate
    "DEXJPUS": "jpy_usd",                      # JPY/USD Exchange Rate (carry trade proxy)
}


# ── Per-stock sector ETF mapping (GICS sectors → SPDR ETFs) ──
SECTOR_ETF_MAP = {
    # Technology
    "AAPL": "XLK", "MSFT": "XLK", "NVDA": "XLK", "AVGO": "XLK", "ORCL": "XLK", "CRM": "XLK",
    "AMD": "XLK", "INTC": "XLK", "QCOM": "XLK", "ADBE": "XLK", "CSCO": "XLK", "IBM": "XLK",
    "TXN": "XLK", "NOW": "XLK", "AMAT": "XLK", "MU": "XLK", "LRCX": "XLK", "KLAC": "XLK",
    "SNPS": "XLK", "CDNS": "XLK", "PANW": "XLK", "CRWD": "XLK", "FTNT": "XLK", "PYPL": "XLK",
    # Communication Services
    "GOOGL": "XLC", "META": "XLC", "NFLX": "XLC", "DIS": "XLC",
    # Consumer Discretionary
    "AMZN": "XLY", "TSLA": "XLY", "MCD": "XLY", "NKE": "XLY", "SBUX": "XLY",
    "TGT": "XLY", "HD": "XLY", "LOW": "XLY", "ABNB": "XLY", "UBER": "XLY",
    # Finance
    "JPM": "XLF", "V": "XLF", "MA": "XLF", "BAC": "XLF", "WFC": "XLF", "GS": "XLF",
    "MS": "XLF", "AXP": "XLF", "BLK": "XLF", "SCHW": "XLF", "C": "XLF", "SQ": "XLF", "COIN": "XLF",
    # Healthcare
    "JNJ": "XLV", "UNH": "XLV", "LLY": "XLV", "PFE": "XLV", "ABBV": "XLV", "MRK": "XLV",
    "TMO": "XLV", "ABT": "XLV", "DHR": "XLV", "BMY": "XLV", "AMGN": "XLV",
    # Consumer Staples
    "PG": "XLP", "KO": "XLP", "PEP": "XLP", "COST": "XLP", "WMT": "XLP",
    # Energy
    "XOM": "XLE", "CVX": "XLE", "COP": "XLE", "SLB": "XLE", "EOG": "XLE",
    # Industrials
    "CAT": "XLI", "DE": "XLI", "HON": "XLI", "UNP": "XLI", "RTX": "XLI",
    "BA": "XLI", "GE": "XLI", "LMT": "XLI", "MMM": "XLI",
    # ETFs
    "SPY": "SPY", "QQQ": "XLK", "IWM": "IWM", "DIA": "DIA",
    "XLF": "XLF", "XLE": "XLE", "XLK": "XLK", "XLV": "XLV", "XBI": "XLV",
}

# ── Per-stock industry commodity mapping ──
INDUSTRY_COMMODITY_MAP = {
    # Energy → Natural Gas
    "XOM": "NG=F", "CVX": "NG=F", "COP": "NG=F", "SLB": "NG=F", "EOG": "NG=F", "XLE": "NG=F",
    # Industrials → Copper
    "CAT": "HG=F", "DE": "HG=F", "HON": "HG=F", "UNP": "HG=F", "RTX": "HG=F",
    "BA": "HG=F", "GE": "HG=F", "LMT": "HG=F", "MMM": "HG=F", "XLI": "HG=F",
    # Crypto-exposed → Bitcoin
    "COIN": "BTC-USD", "SQ": "BTC-USD", "PYPL": "BTC-USD",
}

# ── Additional data tickers for v8.0 features ──
# Market breadth/rotation tickers + stock-specific drivers
BREADTH_TICKERS = ["^SOX", "IWM"]  # SOX semiconductor index, small cap
DRIVER_TICKERS = [
    "IGV",      # iShares Expanded Tech-Software ETF
    "HACK",     # ETFMG Prime Cyber Security ETF
    "KRE",      # SPDR S&P Regional Banking ETF
    "ITA",      # iShares U.S. Aerospace & Defense ETF
    "XOP",      # SPDR S&P Oil & Gas Exploration & Production ETF
    "IBB",      # iShares Biotechnology ETF
    "XHB",      # SPDR S&P Homebuilders ETF
    "XRT",      # SPDR S&P Retail ETF
    "LIT",      # Global X Lithium & Battery Tech ETF
    "ETH-USD",  # Ethereum (crypto sentiment)
    "DBA",      # Invesco DB Agriculture Fund
    "IYT",      # iShares U.S. Transportation ETF
    "XLB",      # Materials Select Sector SPDR
]

# ── Additional data tickers for v9.0 features ──
# Tail risk, style rotation, risk appetite
TAIL_RISK_TICKERS = ["^SKEW"]  # CBOE SKEW Index (options tail risk pricing)
STYLE_ROTATION_TICKERS = ["IWF", "IWD", "XLY", "XLP"]  # Growth/Value, Discretionary/Staples
# (ALL_DATA_TICKERS is defined after STOCK_SPECIFIC_DRIVERS below) + BREADTH_TICKERS + DRIVER_TICKERS

# ── Stock-specific driver mapping ──
# Each stock maps to 2 driving assets (primary, secondary) that capture
# business-specific factors not already in sector ETF or commodity mappings.
# Researched per-company based on revenue drivers, supply chain, and market dynamics.
STOCK_SPECIFIC_DRIVERS = {
    # ── Semiconductors → SOX index + sub-sector ──
    "AAPL": ("^SOX", "XRT"),     # Apple: chip supply chain + consumer retail
    "NVDA": ("^SOX", "BTC-USD"), # NVIDIA: chips + AI/crypto compute demand
    "AMD": ("^SOX", "QQQ"),      # AMD: chips + tech ecosystem
    "INTC": ("^SOX", "QQQ"),     # Intel: chips + PC/server market
    "AVGO": ("^SOX", "QQQ"),     # Broadcom: chips + infrastructure
    "QCOM": ("^SOX", "QQQ"),     # Qualcomm: chips + mobile
    "TXN": ("^SOX", "XLI"),      # Texas Instruments: chips + industrial end-markets
    "AMAT": ("^SOX", "QQQ"),     # Applied Materials: chip equipment
    "MU": ("^SOX", "QQQ"),       # Micron: memory/storage chips
    "LRCX": ("^SOX", "QQQ"),     # Lam Research: chip equipment
    "KLAC": ("^SOX", "QQQ"),     # KLA Corp: chip inspection equipment
    "SNPS": ("^SOX", "IGV"),     # Synopsys: EDA software + chips
    "CDNS": ("^SOX", "IGV"),     # Cadence: EDA software + chips
    # ── Software → IGV (software ETF) + tech ──
    "MSFT": ("IGV", "QQQ"),      # Microsoft: enterprise software + cloud
    "ORCL": ("IGV", "QQQ"),      # Oracle: enterprise DB + cloud
    "CRM": ("IGV", "QQQ"),       # Salesforce: CRM + enterprise cloud
    "ADBE": ("IGV", "QQQ"),      # Adobe: creative/marketing software
    "NOW": ("IGV", "QQQ"),       # ServiceNow: IT workflow automation
    # ── Cybersecurity → HACK + tech ──
    "PANW": ("HACK", "QQQ"),     # Palo Alto: network security
    "CRWD": ("HACK", "QQQ"),     # CrowdStrike: endpoint security
    "FTNT": ("HACK", "QQQ"),     # Fortinet: network security
    # ── Communication/Media → consumer + tech ──
    "GOOGL": ("IGV", "QQQ"),     # Google: ad tech + cloud
    "META": ("IGV", "QQQ"),      # Meta: ad tech + social
    "NFLX": ("XRT", "QQQ"),      # Netflix: consumer streaming + tech
    "DIS": ("XRT", "QQQ"),       # Disney: consumer media + streaming
    # ── Consumer Tech / E-commerce ──
    "AMZN": ("XRT", "QQQ"),      # Amazon: e-commerce + cloud
    "TSLA": ("LIT", "QQQ"),      # Tesla: EV (lithium) + tech
    # ── Legacy Tech ──
    "CSCO": ("IGV", "QQQ"),      # Cisco: networking + enterprise
    "IBM": ("IGV", "QQQ"),       # IBM: enterprise IT + consulting
    # ── Fintech / Crypto ──
    "PYPL": ("IGV", "BTC-USD"),  # PayPal: fintech + crypto
    "SQ": ("IGV", "ETH-USD"),    # Block: fintech + crypto
    "COIN": ("BTC-USD", "ETH-USD"), # Coinbase: pure crypto exchange
    # ── Banks → KRE (regional banks) + rates ──
    "JPM": ("KRE", "^TNX"),      # JPMorgan: banking + rate sensitivity
    "BAC": ("KRE", "^TNX"),      # Bank of America: banking + rates
    "WFC": ("KRE", "^TNX"),      # Wells Fargo: banking + rates
    "GS": ("KRE", "^TNX"),       # Goldman Sachs: banking + rates
    "MS": ("KRE", "^TNX"),       # Morgan Stanley: banking + rates
    "C": ("KRE", "^TNX"),        # Citigroup: banking + rates
    "SCHW": ("KRE", "^TNX"),     # Schwab: brokerage + rates
    # ── Payment Networks ──
    "V": ("XLF", "QQQ"),         # Visa: payments + consumer spend
    "MA": ("XLF", "QQQ"),        # Mastercard: payments + consumer spend
    "AXP": ("XLF", "XRT"),       # AmEx: payments + retail spend
    "BLK": ("XLF", "QQQ"),       # BlackRock: asset management + markets
    # ── Healthcare / Pharma → IBB (biotech) + sector ──
    "JNJ": ("IBB", "XLV"),       # J&J: pharma + diversified healthcare
    "UNH": ("XLV", "QQQ"),       # UnitedHealth: insurance + tech
    "LLY": ("IBB", "XBI"),       # Eli Lilly: pharma + biotech
    "PFE": ("IBB", "XBI"),       # Pfizer: pharma + vaccines
    "ABBV": ("IBB", "XBI"),      # AbbVie: pharma + biotech
    "MRK": ("IBB", "XBI"),       # Merck: pharma + biotech
    "TMO": ("IBB", "XLV"),       # Thermo Fisher: lab equipment
    "ABT": ("IBB", "XLV"),       # Abbott: medical devices + diagnostics
    "DHR": ("IBB", "XLV"),       # Danaher: life sciences + diagnostics
    "BMY": ("IBB", "XBI"),       # Bristol-Myers: pharma + immuno-oncology
    "AMGN": ("IBB", "XBI"),      # Amgen: biotech + pharma
    # ── Consumer Staples → XLP + retail ──
    "PG": ("XLP", "XRT"),        # P&G: staples + consumer
    "KO": ("XLP", "XRT"),        # Coca-Cola: beverages + consumer
    "PEP": ("XLP", "XRT"),       # PepsiCo: beverages/snacks + consumer
    "COST": ("XRT", "XLP"),      # Costco: retail + staples
    "WMT": ("XRT", "XLP"),       # Walmart: retail + staples
    # ── Consumer Discretionary ──
    "MCD": ("XRT", "XLP"),       # McDonald's: QSR + consumer staples-like
    "NKE": ("XRT", "XLY"),       # Nike: athletic retail + consumer
    "SBUX": ("XRT", "XLY"),      # Starbucks: QSR + consumer
    "TGT": ("XRT", "XLY"),       # Target: discount retail + consumer
    "HD": ("XHB", "XRT"),        # Home Depot: housing + retail
    "LOW": ("XHB", "XRT"),       # Lowe's: housing + retail
    # ── Energy → XOP (exploration) + crude oil ──
    "XOM": ("XOP", "CL=F"),      # ExxonMobil: integrated oil + exploration
    "CVX": ("XOP", "CL=F"),      # Chevron: integrated oil + exploration
    "COP": ("XOP", "CL=F"),      # ConocoPhillips: E&P focused
    "SLB": ("XOP", "CL=F"),      # Schlumberger: oilfield services
    "EOG": ("XOP", "CL=F"),      # EOG Resources: E&P focused
    # ── Industrial → sector-specific ETFs ──
    "CAT": ("XLI", "HG=F"),      # Caterpillar: construction + copper/materials
    "DE": ("XLI", "DBA"),        # Deere: agriculture equipment + ag commodities
    "HON": ("XLI", "ITA"),       # Honeywell: aerospace + defense
    "UNP": ("XLI", "IYT"),       # Union Pacific: railroad + transport
    "RTX": ("ITA", "XLI"),       # RTX: defense + aerospace
    "BA": ("ITA", "XLI"),        # Boeing: aerospace + defense
    "GE": ("ITA", "XLI"),        # GE Aerospace: engines + defense
    "LMT": ("ITA", "XLI"),       # Lockheed Martin: defense
    "MMM": ("XLI", "XLB"),       # 3M: industrials + materials
    # ── Travel / Gig Economy ──
    "ABNB": ("XRT", "QQQ"),      # Airbnb: travel/consumer + tech
    "UBER": ("XRT", "QQQ"),      # Uber: mobility/consumer + tech
    # ── ETFs → cross-asset ──
    "SPY": ("QQQ", "IWM"),       # S&P 500 vs NASDAQ + small cap
    "QQQ": ("^SOX", "IGV"),      # NASDAQ vs semis + software
    "IWM": ("SPY", "KRE"),       # Russell 2000 vs S&P + regional banks
    "DIA": ("SPY", "XLI"),       # Dow Jones vs S&P + industrials
    "XLF": ("KRE", "^TNX"),      # Financials vs regional banks + rates
    "XLE": ("XOP", "CL=F"),      # Energy vs exploration + oil
    "XLK": ("^SOX", "IGV"),      # Tech vs semis + software
    "XLV": ("IBB", "XBI"),       # Healthcare vs biotech
    "XBI": ("IBB", "XLV"),       # Biotech vs pharma + healthcare
}

# ── Aggregate all data tickers ──
ALL_DATA_TICKERS = (
    MACRO_TICKERS + CREDIT_TICKERS + EXTRA_MACRO_TICKERS + SECTOR_ETFS
    + BREADTH_TICKERS + DRIVER_TICKERS + TAIL_RISK_TICKERS + STYLE_ROTATION_TICKERS
)

# ── Architecture hyperparameters ──
LOOKBACK_WINDOW = 60       # 60 trading days (~3 months) lookback
FORECAST_HORIZON = 60      # 60 trading days forecast
D_MODEL = 128              # Transformer hidden dimension
N_HEADS = 8                # Attention heads
N_LAYERS = 3               # Transformer encoder layers
D_FF = 256                 # Feed-forward dimension
DROPOUT = 0.15             # Slightly lower dropout for per-stock (less data → less regularization needed)

# ── Per-stock training hyperparameters (tuned for L4 GPU) ──
BATCH_SIZE = 64            # L4 has 24GB VRAM — can handle larger batches
EPOCHS = 80                # More epochs per stock (smaller dataset converges differently)
LEARNING_RATE = 5e-4       # Slightly higher LR for per-stock (less data noise to average over)
WARMUP_EPOCHS = 5          # LR warmup
WEIGHT_DECAY = 5e-4        # Less regularization for per-stock
PATIENCE = 20              # More patience — per-stock loss curves are noisier
TRAIN_SPLIT = 0.70         # 70% train
VAL_SPLIT = 0.85           # Of remaining 30%, 15% val + 15% test
DATA_YEARS = 10            # Years of historical data per stock
MIN_SAMPLES = 500          # Skip stocks with too few samples

# ── Reproducibility ──
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[CONFIG] {len(SCREENER_SYMBOLS)} stocks, {len(ALL_DATA_TICKERS)} data tickers ({len(MACRO_TICKERS)} macro + {len(CREDIT_TICKERS)} credit + {len(EXTRA_MACRO_TICKERS)} extra + {len(SECTOR_ETFS)} sector + {len(BREADTH_TICKERS)} breadth + {len(DRIVER_TICKERS)} driver + {len(TAIL_RISK_TICKERS)} tail-risk + {len(STYLE_ROTATION_TICKERS)} style-rotation)")
print(f"[CONFIG] lookback={LOOKBACK_WINDOW}, horizon={FORECAST_HORIZON}")
print(f"[CONFIG] d_model={D_MODEL}, layers={N_LAYERS}, heads={N_HEADS}, dropout={DROPOUT}")
print(f"[CONFIG] Per-stock: epochs={EPOCHS}, batch={BATCH_SIZE}, lr={LEARNING_RATE}, patience={PATIENCE}")
print(f"[DEVICE] {device}" + (f" — {torch.cuda.get_device_name(0)}" if device.type == 'cuda' else ""))

In [ ]:
# Cell 4: Feature Engineering
# Computes 126 features from OHLCV data + macro indicators + sector/credit/commodity + FRED macro data

import pandas as pd
import yfinance as yf
from typing import Dict, List, Optional, Tuple

def compute_features(df: pd.DataFrame, macro_df: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    """
    Compute 154 features from OHLCV data plus macro indicators.
    Mirrors the TypeScript feature engineering in src/lib/itransformer-features.ts.

    Features 0-82: Original 83 features (OHLCV technicals + macro)
    Features 83-107: 25 features (relative strength, advanced volume,
                     price structure, statistical regime, intermarket)
    Features 108-119: 12 features (sector ETF relative strength, credit market,
                      VIX9D term structure, industry commodity, copper/gold, BTC)
    Features 120-125: 6 features (FRED: HY spread, yield curve, breakeven inflation,
                      2Y Treasury, jobless claims z-score, consumer sentiment change)
    Features 126-131: 6 features (gamma squeeze proxies: volume acceleration,
                      price-volume momentum, range expansion, gap acceleration,
                      squeeze breakout signal, volume-price impact)
    Features 132-135: 4 features (market breadth: tech rotation, small cap rotation,
                      semiconductor momentum, biotech momentum)
    Features 136-139: 4 features (sentiment proxies: realized/implied vol ratio,
                      VIX-SPY correlation, credit momentum, fear composite)
    Features 140-143: 4 features (stock-specific drivers: primary/secondary
                      driver returns and correlations per company)
    Features 144-145: 2 features (FRED extended: financial stress index,
                      10Y-3M yield spread)
    Features 146-147: 2 features (FRED rates: fed funds rate, fed funds change)
    Features 148-149: 2 features (FRED FX: JPY/USD change, JPY/USD z-score)
    Features 150-151: 2 features (tail risk: SKEW level, SKEW z-score)
    Feature 152: value/growth rotation (IWF vs IWD 20d spread)
    Feature 153: risk appetite (XLY vs XLP 20d spread)
    """
    feat = pd.DataFrame(index=df.index)
    close = df["Close"].squeeze()
    high = df["High"].squeeze()
    low = df["Low"].squeeze()
    volume = df["Volume"].squeeze()
    open_ = df["Open"].squeeze()

    # ── Moving Averages (10 features: 0-9) ──
    for p in [5, 10, 20, 50, 200]:
        sma = close.rolling(p).mean()
        feat[f"price_vs_sma_{p}_pct"] = ((close - sma) / sma) * 100

    for p in [5, 12, 26]:
        ema_val = close.ewm(span=p, adjust=False).mean()
        feat[f"price_vs_ema_{p}_pct"] = ((close - ema_val) / ema_val) * 100

    feat["sma_20_50_cross"] = (close.rolling(20).mean() > close.rolling(50).mean()).astype(float)
    feat["sma_50_200_cross"] = (close.rolling(50).mean() > close.rolling(200).mean()).astype(float)

    # ── RSI (3 features: 10-12) ──
    for p in [7, 14, 21]:
        delta = close.diff()
        gain = delta.where(delta > 0, 0).rolling(p).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(p).mean()
        rs = gain / (loss + 1e-10)
        feat[f"rsi_{p}"] = 100 - 100 / (1 + rs)

    # ── MACD (2 features: 13-14) ──
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    macd_line = ema12 - ema26
    macd_signal = macd_line.ewm(span=9, adjust=False).mean()
    feat["macd_histogram"] = (macd_line - macd_signal) / close * 100
    feat["macd_cross_above"] = ((macd_line > macd_signal) & (macd_line.shift(1) <= macd_signal.shift(1))).astype(float)

    # ── Bollinger Bands (2 features: 15-16) ──
    bb_sma = close.rolling(20).mean()
    bb_std = close.rolling(20).std()
    feat["bb_width"] = (4 * bb_std / bb_sma) * 100
    feat["bb_pctb"] = (close - (bb_sma - 2 * bb_std)) / (4 * bb_std + 1e-10)

    # ── ATR (2 features: 17-18) ──
    tr1 = high - low
    tr2 = (high - close.shift(1)).abs()
    tr3 = (low - close.shift(1)).abs()
    tr = np.maximum(np.maximum(tr1, tr2), tr3)
    atr7 = tr.rolling(7).mean()
    atr14 = tr.rolling(14).mean()
    feat["atr_7_pct"] = (atr7 / close) * 100
    feat["atr_14_pct"] = (atr14 / close) * 100

    # ── Volume (5 features: 19-23) ──
    feat["volume_ratio_5_20"] = volume.rolling(5).mean() / (volume.rolling(20).mean() + 1)
    feat["relative_volume"] = volume / (volume.rolling(20).mean() + 1)
    obv = (np.sign(close.diff()) * volume).cumsum()
    obv_norm = (obv - obv.rolling(20).mean()) / (obv.rolling(20).std() + 1e-10)
    feat["obv_zscore"] = obv_norm
    clv = ((close - low) - (high - close)) / (high - low + 1e-10)
    feat["cmf_20"] = (clv * volume).rolling(20).sum() / (volume.rolling(20).sum() + 1)
    feat["volume_zscore_20"] = (volume - volume.rolling(20).mean()) / (volume.rolling(20).std() + 1e-10)

    # ── Stochastic (2 features: 24-25) ──
    low14 = low.rolling(14).min()
    high14 = high.rolling(14).max()
    feat["stoch_k"] = ((close - low14) / (high14 - low14 + 1e-10)) * 100
    feat["stoch_d"] = feat["stoch_k"].rolling(3).mean()

    # ── Williams %R (1 feature: 26) ──
    feat["williams_r"] = ((high14 - close) / (high14 - low14 + 1e-10)) * -100

    # ── ROC (3 features: 27-29) ──
    for p in [5, 10, 20]:
        feat[f"roc_{p}"] = close.pct_change(p) * 100

    # ── CCI (1 feature: 30) ──
    tp = (high + low + close) / 3
    tp_sma = tp.rolling(20).mean()
    tp_mad = tp.rolling(20).apply(lambda x: np.mean(np.abs(x - np.mean(x))))
    feat["cci_20"] = (tp - tp_sma) / (0.015 * tp_mad + 1e-10)

    # ── Aroon (3 features: 31-33) ──
    feat["aroon_up"] = high.rolling(25).apply(lambda x: x.argmax() / 24 * 100)
    feat["aroon_down"] = low.rolling(25).apply(lambda x: x.argmin() / 24 * 100)
    feat["aroon_oscillator"] = feat["aroon_up"] - feat["aroon_down"]

    # ── Returns (5 features: 34-38) ──
    for p in [1, 5, 10, 20, 60]:
        feat[f"return_{p}d"] = close.pct_change(p)

    # ── Volatility (4 features: 39-42) ──
    log_ret = np.log(close / close.shift(1))
    for p in [5, 10, 20, 60]:
        feat[f"volatility_{p}d"] = log_ret.rolling(p).std() * np.sqrt(252)

    # ── Higher Moments (4 features: 43-46) ──
    feat["skewness_20d"] = log_ret.rolling(20).skew()
    feat["skewness_60d"] = log_ret.rolling(60).skew()
    feat["kurtosis_20d"] = log_ret.rolling(20).kurt()
    feat["kurtosis_60d"] = log_ret.rolling(60).kurt()

    # ── Autocorrelation (3 features: 47-49) ──
    for lag in [1, 3, 5]:
        feat[f"autocorr_lag_{lag}"] = log_ret.rolling(30).apply(
            lambda x: x.autocorr(lag) if len(x) >= lag + 2 else 0
        )

    # ── Z-Scores (4 features: 50-53) ──
    for p in [20, 50, 100, 200]:
        roll_mean = close.rolling(p).mean()
        roll_std = close.rolling(p).std()
        feat[f"zscore_{p}"] = (close - roll_mean) / (roll_std + 1e-10)

    # ── Percentile Ranks (3 features: 54-56) ──
    for p in [20, 60, 252]:
        feat[f"percentile_rank_{p}d"] = close.rolling(p).apply(
            lambda x: (x < x.iloc[-1]).sum() / len(x) * 100 if len(x) == p else 50
        )

    # ── Max Drawdown (2 features: 57-58) ──
    for p in [20, 60]:
        rolling_max = close.rolling(p).max()
        feat[f"max_drawdown_{p}d"] = (close - rolling_max) / (rolling_max + 1e-10)

    # ── Up/Down Ratios (2 features: 59-60) ──
    for p in [10, 20]:
        feat[f"up_ratio_{p}d"] = (close.diff() > 0).rolling(p).mean()

    # ── Gap Features (2 features: 61-62) ──
    gap = (open_ - close.shift(1)) / (close.shift(1) + 1e-10)
    feat["avg_gap_20d"] = gap.rolling(20).mean()
    feat["gap_frequency_20d"] = (gap.abs() > 0.01).rolling(20).mean()

    # ── Calendar Features (5 features: 63-67) ──
    dates = pd.to_datetime(df.index)
    feat["day_of_week"] = dates.dayofweek / 4
    feat["month_sin"] = np.sin(2 * np.pi * dates.month / 12)
    feat["month_cos"] = np.cos(2 * np.pi * dates.month / 12)
    feat["is_quarter_end"] = dates.month.isin([3, 6, 9, 12]).astype(float)
    feat["is_opex_week"] = ((dates.day >= 15) & (dates.day <= 21)).astype(float)

    # ── Trend Strength (3 features: 68-70) ──
    feat["price_slope_20"] = close.rolling(20).apply(
        lambda x: np.polyfit(range(len(x)), x / x.iloc[0], 1)[0] if len(x) == 20 else 0
    )
    feat["price_slope_50"] = close.rolling(50).apply(
        lambda x: np.polyfit(range(len(x)), x / x.iloc[0], 1)[0] if len(x) == 50 else 0
    )
    tenkan = (high.rolling(9).max() + low.rolling(9).min()) / 2
    kijun = (high.rolling(26).max() + low.rolling(26).min()) / 2
    feat["ichimoku_tk_cross"] = (tenkan > kijun).astype(float)

    # ── Volatility Regime (2 features: 71-72) ──
    feat["vol_regime_ratio"] = feat["volatility_20d"] / (feat["volatility_60d"] + 1e-10)
    feat["vol_expanding"] = (feat["volatility_20d"] > feat["volatility_60d"]).astype(float)

    # ── Macro Features (10 features: 73-82) ──
    if macro_df is not None:
        if "^VIX" in macro_df.columns:
            vix = macro_df["^VIX"].reindex(df.index, method="ffill")
            feat["vix_level"] = vix
            feat["vix_change_5d"] = vix.pct_change(5)
            feat["vix_zscore_20"] = (vix - vix.rolling(20).mean()) / (vix.rolling(20).std() + 1e-10)

        if "^VIX" in macro_df.columns and "^VIX3M" in macro_df.columns:
            vix = macro_df["^VIX"].reindex(df.index, method="ffill")
            vix3m = macro_df["^VIX3M"].reindex(df.index, method="ffill")
            feat["vix_term_structure"] = (vix3m - vix) / (vix + 1e-10)

        if "^TNX" in macro_df.columns:
            tnx = macro_df["^TNX"].reindex(df.index, method="ffill")
            feat["treasury_10y"] = tnx
            feat["treasury_change_20d"] = tnx.diff(20)

        if "DX-Y.NYB" in macro_df.columns:
            dxy = macro_df["DX-Y.NYB"].reindex(df.index, method="ffill")
            feat["usd_index"] = dxy
            feat["usd_change_20d"] = dxy.pct_change(20)

        if "GC=F" in macro_df.columns:
            gold = macro_df["GC=F"].reindex(df.index, method="ffill")
            feat["gold_change_20d"] = gold.pct_change(20)

        if "CL=F" in macro_df.columns:
            oil = macro_df["CL=F"].reindex(df.index, method="ffill")
            feat["oil_change_20d"] = oil.pct_change(20)

    # ═══════════════════════════════════════════════════════════════
    # NEW FEATURES (25 features: 83-107)
    # ═══════════════════════════════════════════════════════════════

    # ── Sector Relative Strength (4 features: 83-86) ──
    # Compare stock returns vs SPY (market benchmark) at multiple horizons
    if macro_df is not None and "SPY" in macro_df.columns:
        spy = macro_df["SPY"].reindex(df.index, method="ffill")
        spy_ret_5 = spy.pct_change(5)
        spy_ret_20 = spy.pct_change(20)
        spy_ret_60 = spy.pct_change(60)
        feat["rel_return_vs_spy_5d"] = close.pct_change(5) - spy_ret_5
        feat["rel_return_vs_spy_20d"] = close.pct_change(20) - spy_ret_20
        feat["rel_return_vs_spy_60d"] = close.pct_change(60) - spy_ret_60
        # 20-day rolling correlation with SPY
        spy_log_ret = np.log(spy / spy.shift(1))
        feat["rolling_corr_spy_20d"] = log_ret.rolling(20).corr(spy_log_ret)
    else:
        feat["rel_return_vs_spy_5d"] = 0.0
        feat["rel_return_vs_spy_20d"] = 0.0
        feat["rel_return_vs_spy_60d"] = 0.0
        feat["rolling_corr_spy_20d"] = 0.0

    # ── Cross-Asset Correlations (3 features: 87-89) ──
    if macro_df is not None and "^VIX" in macro_df.columns:
        vix = macro_df["^VIX"].reindex(df.index, method="ffill")
        vix_log_ret = np.log(vix / vix.shift(1))
        feat["rolling_corr_vix_20d"] = log_ret.rolling(20).corr(vix_log_ret)
    else:
        feat["rolling_corr_vix_20d"] = 0.0

    if macro_df is not None and "SPY" in macro_df.columns:
        spy = macro_df["SPY"].reindex(df.index, method="ffill")
        spy_log_ret = np.log(spy / spy.shift(1))
        # Rolling beta to SPY (covariance / variance)
        cov_spy = log_ret.rolling(60).cov(spy_log_ret)
        var_spy = spy_log_ret.rolling(60).var()
        feat["rolling_beta_spy_60d"] = cov_spy / (var_spy + 1e-10)
    else:
        feat["rolling_beta_spy_60d"] = 1.0

    # Volume-price correlation (institutional accumulation signal)
    feat["corr_volume_price_20d"] = log_ret.rolling(20).corr(
        np.log(volume / volume.shift(1).replace(0, 1))
    )

    # ── Advanced Volume (4 features: 90-93) ──
    # Money Flow Index (14-period) — volume-weighted RSI
    typical_price = (high + low + close) / 3
    raw_money_flow = typical_price * volume
    pos_flow = raw_money_flow.where(typical_price > typical_price.shift(1), 0)
    neg_flow = raw_money_flow.where(typical_price < typical_price.shift(1), 0)
    money_ratio = pos_flow.rolling(14).sum() / (neg_flow.rolling(14).sum() + 1e-10)
    feat["mfi_14"] = 100 - 100 / (1 + money_ratio)

    # Accumulation/Distribution line z-score
    ad_line = (clv * volume).cumsum()
    ad_mean = ad_line.rolling(20).mean()
    ad_std = ad_line.rolling(20).std()
    feat["ad_line_zscore"] = (ad_line - ad_mean) / (ad_std + 1e-10)

    # VWAP deviation (typical price vs volume-weighted average)
    vwap_20 = (typical_price * volume).rolling(20).sum() / (volume.rolling(20).sum() + 1)
    feat["vwap_deviation"] = ((close - vwap_20) / vwap_20) * 100

    # Force Index (13-period EMA) — price × volume directional power
    force_raw = close.diff() * volume
    feat["force_index_13"] = force_raw.ewm(span=13, adjust=False).mean() / (close * volume.rolling(20).mean() + 1e-10) * 100

    # ── Price Structure (5 features: 94-98) ──
    # Range position — where price sits within recent high-low range (0-1)
    low_20 = low.rolling(20).min()
    high_20 = high.rolling(20).max()
    feat["range_position_20d"] = (close - low_20) / (high_20 - low_20 + 1e-10)

    low_60 = low.rolling(60).min()
    high_60 = high.rolling(60).max()
    feat["range_position_60d"] = (close - low_60) / (high_60 - low_60 + 1e-10)

    # ATR ratio — short-term vs long-term volatility expansion
    atr60 = tr.rolling(60).mean()
    feat["atr_ratio_7_60"] = atr7 / (atr60 + 1e-10)

    # Consecutive up/down days (momentum persistence)
    up_days = (close.diff() > 0).astype(float)
    consecutive = up_days.copy()
    for i in range(1, len(consecutive)):
        if consecutive.iloc[i] == 1:
            consecutive.iloc[i] = consecutive.iloc[i-1] + 1
        else:
            consecutive.iloc[i] = 0
    feat["consecutive_up_days"] = consecutive / 10.0  # Normalize (10 consecutive days = 1.0)

    # Average candle body ratio over 5 days (buying/selling pressure)
    body = (close - open_).abs()
    wick = high - low + 1e-10
    feat["candle_body_ratio_5d"] = (body / wick).rolling(5).mean()

    # ── Statistical Regime Detection (5 features: 99-103) ──
    # Hurst exponent approximation (trend persistence: >0.5 trending, <0.5 mean-reverting)
    def hurst_rs(x):
        n = len(x)
        if n < 20:
            return 0.5
        mean_x = np.mean(x)
        y = np.cumsum(x - mean_x)
        r = np.max(y) - np.min(y)
        s = np.std(x, ddof=1) + 1e-10
        rs = r / s
        if rs <= 0:
            return 0.5
        return np.log(rs) / np.log(n)
    feat["hurst_exponent"] = log_ret.rolling(100).apply(hurst_rs)

    # Parkinson volatility (uses high-low, more efficient estimator than close-to-close)
    feat["parkinson_vol_20d"] = np.sqrt(
        (1 / (4 * 20 * np.log(2))) * ((np.log(high / low)) ** 2).rolling(20).sum()
    ) * np.sqrt(252)

    # Garman-Klass volatility (uses OHLC, most efficient estimator)
    gk_component = 0.5 * (np.log(high / low)) ** 2 - (2 * np.log(2) - 1) * (np.log(close / open_)) ** 2
    feat["garman_klass_vol_20d"] = np.sqrt(gk_component.rolling(20).mean() * 252)

    # Return consistency (signal-to-noise ratio of returns)
    feat["return_consistency_20d"] = log_ret.rolling(20).mean().abs() / (log_ret.rolling(20).std() + 1e-10)

    # Tail ratio (asymmetry of return distribution tails)
    def tail_ratio(x):
        if len(x) < 20:
            return 1.0
        sorted_x = np.sort(x)
        n = len(sorted_x)
        top_5 = np.mean(sorted_x[int(n * 0.95):]) if int(n * 0.95) < n else sorted_x[-1]
        bottom_5 = np.abs(np.mean(sorted_x[:max(1, int(n * 0.05))])) + 1e-10
        return top_5 / bottom_5
    feat["tail_ratio_20d"] = log_ret.rolling(20).apply(tail_ratio)

    # ── Intermarket Features (4 features: 104-107) ──
    if macro_df is not None and "SPY" in macro_df.columns:
        spy = macro_df["SPY"].reindex(df.index, method="ffill")
        feat["spy_return_5d"] = spy.pct_change(5)
        feat["spy_return_20d"] = spy.pct_change(20)
    else:
        feat["spy_return_5d"] = 0.0
        feat["spy_return_20d"] = 0.0

    if macro_df is not None and "GC=F" in macro_df.columns and "CL=F" in macro_df.columns:
        gold = macro_df["GC=F"].reindex(df.index, method="ffill")
        oil = macro_df["CL=F"].reindex(df.index, method="ffill")
        gold_oil = gold / (oil + 1e-10)
        feat["gold_oil_ratio_change"] = gold_oil.pct_change(20)
    else:
        feat["gold_oil_ratio_change"] = 0.0

    if macro_df is not None and "DX-Y.NYB" in macro_df.columns and "^VIX" in macro_df.columns:
        dxy = macro_df["DX-Y.NYB"].reindex(df.index, method="ffill")
        vix = macro_df["^VIX"].reindex(df.index, method="ffill")
        feat["dxy_vix_interaction"] = dxy.pct_change(5) * vix.pct_change(5) * 100
    else:
        feat["dxy_vix_interaction"] = 0.0

    # ═══════════════════════════════════════════════════════════════
    # INDUSTRY / SECTOR / CREDIT FEATURES (12 features: 108-119)
    # ═══════════════════════════════════════════════════════════════

    # ── Sector ETF Relative Strength (3 features: 108-110) ──
    # Compare stock returns vs its GICS sector ETF
    if macro_df is not None and "SECTOR_ETF" in macro_df.columns:
        sector = macro_df["SECTOR_ETF"].reindex(df.index, method="ffill")
        sector_ret_5 = sector.pct_change(5)
        sector_ret_20 = sector.pct_change(20)
        feat["sector_rel_return_5d"] = close.pct_change(5) - sector_ret_5
        feat["sector_rel_return_20d"] = close.pct_change(20) - sector_ret_20
        sector_log_ret = np.log(sector / sector.shift(1))
        feat["sector_corr_20d"] = log_ret.rolling(20).corr(sector_log_ret)
    else:
        feat["sector_rel_return_5d"] = 0.0
        feat["sector_rel_return_20d"] = 0.0
        feat["sector_corr_20d"] = 0.0

    # ── Credit Market Signals (4 features: 111-114) ──
    if macro_df is not None and "HYG" in macro_df.columns:
        hyg = macro_df["HYG"].reindex(df.index, method="ffill")
        feat["hyg_return_20d"] = hyg.pct_change(20)
    else:
        feat["hyg_return_20d"] = 0.0

    if macro_df is not None and "TLT" in macro_df.columns:
        tlt = macro_df["TLT"].reindex(df.index, method="ffill")
        feat["tlt_return_20d"] = tlt.pct_change(20)
    else:
        feat["tlt_return_20d"] = 0.0

    if macro_df is not None and "HYG" in macro_df.columns and "TLT" in macro_df.columns:
        hyg = macro_df["HYG"].reindex(df.index, method="ffill")
        tlt = macro_df["TLT"].reindex(df.index, method="ffill")
        hyg_tlt_ratio = hyg / (tlt + 1e-10)
        feat["credit_spread_change_20d"] = hyg_tlt_ratio.pct_change(20)
    else:
        feat["credit_spread_change_20d"] = 0.0

    if macro_df is not None and "HYG" in macro_df.columns and "SPY" in macro_df.columns:
        hyg = macro_df["HYG"].reindex(df.index, method="ffill")
        spy = macro_df["SPY"].reindex(df.index, method="ffill")
        feat["hyg_spy_divergence"] = hyg.pct_change(20) - spy.pct_change(20)
    else:
        feat["hyg_spy_divergence"] = 0.0

    # ── VIX 9-Day Term Structure (1 feature: 115) ──
    if macro_df is not None and "^VIX9D" in macro_df.columns and "^VIX" in macro_df.columns:
        vix9d = macro_df["^VIX9D"].reindex(df.index, method="ffill")
        vix = macro_df["^VIX"].reindex(df.index, method="ffill")
        feat["vix_9d_ratio"] = vix9d / (vix + 1e-10)
    else:
        feat["vix_9d_ratio"] = 1.0  # neutral default

    # ── Industry Commodity Sensitivity (2 features: 116-117) ──
    if macro_df is not None and "INDUSTRY_COMMODITY" in macro_df.columns:
        commodity = macro_df["INDUSTRY_COMMODITY"].reindex(df.index, method="ffill")
        commodity_log_ret = np.log(commodity / commodity.shift(1))
        feat["industry_commodity_corr_20d"] = log_ret.rolling(20).corr(commodity_log_ret)
        feat["industry_commodity_return_20d"] = commodity.pct_change(20)
    else:
        feat["industry_commodity_corr_20d"] = 0.0
        feat["industry_commodity_return_20d"] = 0.0

    # ── Copper/Gold Ratio Change (1 feature: 118) ──
    if macro_df is not None and "HG=F" in macro_df.columns and "GC=F" in macro_df.columns:
        copper = macro_df["HG=F"].reindex(df.index, method="ffill")
        gold = macro_df["GC=F"].reindex(df.index, method="ffill")
        cu_au_ratio = copper / (gold + 1e-10)
        feat["copper_gold_ratio_change"] = cu_au_ratio.pct_change(20)
    else:
        feat["copper_gold_ratio_change"] = 0.0

    # ── Bitcoin Sentiment (1 feature: 119) ──
    if macro_df is not None and "BTC-USD" in macro_df.columns:
        btc = macro_df["BTC-USD"].reindex(df.index, method="ffill")
        feat["btc_change_20d"] = btc.pct_change(20)
    else:
        feat["btc_change_20d"] = 0.0


    # ═══════════════════════════════════════════════════════════════
    # FRED MACRO FEATURES (6 features: 120-125)
    # Federal Reserve Economic Data — national US economic indicators
    # ═══════════════════════════════════════════════════════════════

    # ── HY Credit Spread (1 feature: 120) ──
    if macro_df is not None and "FRED_HY_SPREAD" in macro_df.columns:
        hy = macro_df["FRED_HY_SPREAD"].reindex(df.index, method="ffill")
        feat["fred_hy_spread"] = hy
    else:
        feat["fred_hy_spread"] = 0.0

    # ── Yield Curve 10Y-2Y (1 feature: 121) ──
    if macro_df is not None and "FRED_YIELD_CURVE" in macro_df.columns:
        yc = macro_df["FRED_YIELD_CURVE"].reindex(df.index, method="ffill")
        feat["fred_yield_curve"] = yc
    else:
        feat["fred_yield_curve"] = 0.0

    # ── Breakeven Inflation (1 feature: 122) ──
    if macro_df is not None and "FRED_BREAKEVEN" in macro_df.columns:
        bei = macro_df["FRED_BREAKEVEN"].reindex(df.index, method="ffill")
        feat["fred_breakeven_inflation"] = bei
    else:
        feat["fred_breakeven_inflation"] = 0.0

    # ── 2-Year Treasury Yield (1 feature: 123) ──
    if macro_df is not None and "FRED_2Y_YIELD" in macro_df.columns:
        t2y = macro_df["FRED_2Y_YIELD"].reindex(df.index, method="ffill")
        feat["fred_2y_yield"] = t2y
    else:
        feat["fred_2y_yield"] = 0.0

    # ── Initial Jobless Claims Z-Score (1 feature: 124) ──
    # Weekly data forward-filled to daily, then z-scored over 20-day window
    if macro_df is not None and "FRED_JOBLESS_CLAIMS" in macro_df.columns:
        claims = macro_df["FRED_JOBLESS_CLAIMS"].reindex(df.index, method="ffill")
        claims_mean = claims.rolling(20).mean()
        claims_std = claims.rolling(20).std() + 1e-10
        feat["fred_jobless_claims_zscore"] = (claims - claims_mean) / claims_std
    else:
        feat["fred_jobless_claims_zscore"] = 0.0

    # ── Consumer Sentiment Change (1 feature: 125) ──
    # Monthly data forward-filled to daily, 20-day pct change
    if macro_df is not None and "FRED_CONSUMER_SENTIMENT" in macro_df.columns:
        sent = macro_df["FRED_CONSUMER_SENTIMENT"].reindex(df.index, method="ffill")
        feat["fred_consumer_sentiment_change"] = sent.pct_change(20)
    else:
        feat["fred_consumer_sentiment_change"] = 0.0

    # ═══════════════════════════════════════════════════════════════
    # GAMMA SQUEEZE PROXY FEATURES (6 features: 126-131)
    # Market makers short gamma on short-dated OTM options. When stock
    # nears strikes, hedging amplifies price moves. These features detect
    # the OHLCV signatures of potential gamma squeezes without requiring
    # historical options data (which is not freely available).
    # ═══════════════════════════════════════════════════════════════

    # Volume acceleration — sudden spike in volume (3d avg / 10d avg)
    # High values indicate unusual activity, potentially from options hedging flows
    vol3 = volume.rolling(3).mean()
    vol10 = volume.rolling(10).mean()
    feat["volume_acceleration_3_10"] = vol3 / (vol10 + 1)

    # Price-volume momentum — large price moves amplified by high volume
    # Captures the "mechanical move" signature of gamma squeezes
    feat["price_volume_momentum_5d"] = close.pct_change(5) * (vol3 / (vol10 + 1))

    # Range expansion ratio — intraday range vs 20d average
    # Gamma squeezes produce unusually wide intraday ranges
    intraday_range = (high - low) / (close + 1e-10)
    avg_range_20 = intraday_range.rolling(20).mean()
    feat["range_expansion_ratio"] = intraday_range / (avg_range_20 + 1e-10)

    # Gap acceleration — gap frequency increasing (10d vs 20d)
    # Pre-squeeze behavior often shows increasing overnight gaps
    gap_abs = ((open_ - close.shift(1)) / (close.shift(1) + 1e-10)).abs()
    gap_freq_10 = (gap_abs > 0.01).rolling(10).mean()
    gap_freq_20 = (gap_abs > 0.01).rolling(20).mean()
    feat["gap_acceleration_10_20"] = gap_freq_10 - gap_freq_20

    # Squeeze breakout signal — close above upper Bollinger Band + high volume
    # Binary signal detecting the breakout phase of a squeeze
    bb_upper = bb_sma + 2 * bb_std
    above_bb = (close > bb_upper).astype(float)
    vol_zscore = (volume - volume.rolling(20).mean()) / (volume.rolling(20).std() + 1e-10)
    feat["squeeze_breakout_signal"] = above_bb * (vol_zscore > 1.5).astype(float)

    # Volume-price impact — how much price moves per unit of relative volume
    # Low float / high gamma stocks show disproportionate price impact
    abs_return_1d = close.pct_change(1).abs()
    rel_vol = volume / (volume.rolling(20).mean() + 1)
    feat["volume_price_impact_1d"] = abs_return_1d / (rel_vol + 1e-10)

    # ═══════════════════════════════════════════════════════════════
    # MARKET BREADTH & ROTATION (4 features: 132-135)
    # Capture style rotation, sector rotation, and industry-cycle signals
    # that affect individual stock price dynamics.
    # ═══════════════════════════════════════════════════════════════

    # Tech rotation — QQQ vs SPY relative performance (growth vs value)
    if macro_df is not None and "QQQ" in macro_df.columns and "SPY" in macro_df.columns:
        qqq = macro_df["QQQ"].reindex(df.index, method="ffill")
        spy = macro_df["SPY"].reindex(df.index, method="ffill")
        feat["tech_rotation_20d"] = qqq.pct_change(20) - spy.pct_change(20)
    else:
        feat["tech_rotation_20d"] = 0.0

    # Small cap rotation — IWM vs SPY (risk appetite / breadth)
    if macro_df is not None and "IWM" in macro_df.columns and "SPY" in macro_df.columns:
        iwm = macro_df["IWM"].reindex(df.index, method="ffill")
        spy = macro_df["SPY"].reindex(df.index, method="ffill")
        feat["small_cap_rotation_20d"] = iwm.pct_change(20) - spy.pct_change(20)
    else:
        feat["small_cap_rotation_20d"] = 0.0

    # Semiconductor momentum — SOX index 20d return (chip cycle indicator)
    if macro_df is not None and "^SOX" in macro_df.columns:
        sox = macro_df["^SOX"].reindex(df.index, method="ffill")
        feat["sox_momentum_20d"] = sox.pct_change(20)
    else:
        feat["sox_momentum_20d"] = 0.0

    # Biotech momentum — XBI 20d return (healthcare risk appetite)
    if macro_df is not None and "XBI" in macro_df.columns:
        xbi = macro_df["XBI"].reindex(df.index, method="ffill")
        feat["xbi_momentum_20d"] = xbi.pct_change(20)
    else:
        feat["xbi_momentum_20d"] = 0.0

    # ═══════════════════════════════════════════════════════════════
    # SENTIMENT PROXIES (4 features: 136-139)
    # Market sentiment indicators derived from cross-asset relationships
    # ═══════════════════════════════════════════════════════════════

    # Realized vs Implied Volatility ratio — vol risk premium proxy
    # Low ratio = cheap protection = complacency, High ratio = expensive protection
    if macro_df is not None and "^VIX" in macro_df.columns:
        vix = macro_df["^VIX"].reindex(df.index, method="ffill")
        hv20_ann = log_ret.rolling(20).std() * np.sqrt(252)
        feat["realized_implied_vol_ratio"] = hv20_ann / (vix / 100 + 1e-10)
    else:
        feat["realized_implied_vol_ratio"] = 1.0

    # VIX-SPY short-term correlation (10d) — fear intensity
    # Strong negative correlation = normal hedging; breakdown = panic
    if macro_df is not None and "^VIX" in macro_df.columns and "SPY" in macro_df.columns:
        vix = macro_df["^VIX"].reindex(df.index, method="ffill")
        spy = macro_df["SPY"].reindex(df.index, method="ffill")
        vix_ret = vix.pct_change(1)
        spy_ret = spy.pct_change(1)
        feat["vix_spy_short_corr_10d"] = vix_ret.rolling(10).corr(spy_ret)
    else:
        feat["vix_spy_short_corr_10d"] = -0.7  # typical negative correlation

    # Credit momentum 10d — fast HYG signal (risk appetite barometer)
    if macro_df is not None and "HYG" in macro_df.columns:
        hyg = macro_df["HYG"].reindex(df.index, method="ffill")
        feat["credit_momentum_10d"] = hyg.pct_change(10)
    else:
        feat["credit_momentum_10d"] = 0.0

    # Fear composite — multi-signal composite indicator
    # Combines VIX z-score and credit spread change into single fear metric
    if macro_df is not None and "^VIX" in macro_df.columns:
        vix = macro_df["^VIX"].reindex(df.index, method="ffill")
        vix_z = (vix - vix.rolling(20).mean()) / (vix.rolling(20).std() + 1e-10)
        if "HYG" in macro_df.columns and "TLT" in macro_df.columns:
            hyg = macro_df["HYG"].reindex(df.index, method="ffill")
            tlt = macro_df["TLT"].reindex(df.index, method="ffill")
            credit_chg = (hyg / (tlt + 1e-10)).pct_change(20)
            feat["fear_composite"] = vix_z * (1 - credit_chg)
        else:
            feat["fear_composite"] = vix_z
    else:
        feat["fear_composite"] = 0.0

    # ═══════════════════════════════════════════════════════════════
    # STOCK-SPECIFIC DRIVER FEATURES (4 features: 140-143)
    # Per-stock driving assets based on each company's business model,
    # supply chain, and market dynamics. Each stock maps to 2 unique
    # drivers via STOCK_SPECIFIC_DRIVERS mapping.
    # ═══════════════════════════════════════════════════════════════

    if macro_df is not None and "STOCK_DRIVER_1" in macro_df.columns:
        driver1 = macro_df["STOCK_DRIVER_1"].reindex(df.index, method="ffill")
        feat["stock_driver_1_return_20d"] = driver1.pct_change(20)
        driver1_log_ret = np.log(driver1 / driver1.shift(1))
        feat["stock_driver_1_corr_20d"] = log_ret.rolling(20).corr(driver1_log_ret)
    else:
        feat["stock_driver_1_return_20d"] = 0.0
        feat["stock_driver_1_corr_20d"] = 0.0

    if macro_df is not None and "STOCK_DRIVER_2" in macro_df.columns:
        driver2 = macro_df["STOCK_DRIVER_2"].reindex(df.index, method="ffill")
        feat["stock_driver_2_return_20d"] = driver2.pct_change(20)
        driver2_log_ret = np.log(driver2 / driver2.shift(1))
        feat["stock_driver_2_corr_20d"] = log_ret.rolling(20).corr(driver2_log_ret)
    else:
        feat["stock_driver_2_return_20d"] = 0.0
        feat["stock_driver_2_corr_20d"] = 0.0

    # ═══════════════════════════════════════════════════════════════
    # FRED EXTENDED FEATURES (2 features: 144-145)
    # Additional FRED macro indicators for regime detection
    # ═══════════════════════════════════════════════════════════════

    # St. Louis Fed Financial Stress Index — weekly composite of 18 financial
    # indicators (interest rates, yield spreads, volatility). Zero = normal,
    # positive = above-average stress, negative = below-average.
    if macro_df is not None and "FRED_FINANCIAL_STRESS" in macro_df.columns:
        stress = macro_df["FRED_FINANCIAL_STRESS"].reindex(df.index, method="ffill")
        feat["fred_financial_stress"] = stress
    else:
        feat["fred_financial_stress"] = 0.0

    # 10Y-3M Treasury spread — alternative recession indicator
    # More sensitive than 10Y-2Y; inversion has preceded every US recession since 1970.
    if macro_df is not None and "FRED_T10Y3M_SPREAD" in macro_df.columns:
        t10y3m = macro_df["FRED_T10Y3M_SPREAD"].reindex(df.index, method="ffill")
        feat["fred_t10y3m_spread"] = t10y3m
    else:
        feat["fred_t10y3m_spread"] = 0.0

    # ── Features 146-147: Fed Funds Rate ──
    if macro_df is not None and "fed_funds_rate" in macro_df.columns:
        ffr = macro_df["fed_funds_rate"].reindex(df.index, method="ffill").fillna(0)
        feat["fed_funds_rate"] = ffr
        feat["fed_funds_rate_change_20d"] = ffr.diff(20).fillna(0)
    else:
        feat["fed_funds_rate"] = 0.0
        feat["fed_funds_rate_change_20d"] = 0.0

    # ── Features 148-149: JPY/USD (carry trade proxy) ──
    if macro_df is not None and "jpy_usd" in macro_df.columns:
        jpy = macro_df["jpy_usd"].reindex(df.index, method="ffill").fillna(0)
        jpy_safe = jpy.replace(0, np.nan)
        feat["jpy_usd_change_20d"] = jpy_safe.pct_change(20).fillna(0)
        jpy_mean = jpy_safe.rolling(20, min_periods=5).mean()
        jpy_std = jpy_safe.rolling(20, min_periods=5).std().replace(0, 1e-10)
        feat["jpy_usd_zscore_20"] = ((jpy_safe - jpy_mean) / jpy_std).fillna(0)
    else:
        feat["jpy_usd_change_20d"] = 0.0
        feat["jpy_usd_zscore_20"] = 0.0

    # ── Features 150-151: CBOE SKEW Index (tail risk) ──
    if macro_df is not None and "^SKEW" in macro_df.columns:
        skew_raw = macro_df["^SKEW"].reindex(df.index, method="ffill").fillna(0)
        feat["skew_level"] = skew_raw
        skew_safe = skew_raw.replace(0, np.nan)
        skew_mean = skew_safe.rolling(20, min_periods=5).mean()
        skew_std = skew_safe.rolling(20, min_periods=5).std().replace(0, 1e-10)
        feat["skew_zscore_20"] = ((skew_safe - skew_mean) / skew_std).fillna(0)
    else:
        feat["skew_level"] = 0.0
        feat["skew_zscore_20"] = 0.0

    # ── Feature 152: Value/Growth Rotation (IWF vs IWD) ──
    if macro_df is not None and "IWF" in macro_df.columns and "IWD" in macro_df.columns:
        iwf = macro_df["IWF"].reindex(df.index, method="ffill")
        iwd = macro_df["IWD"].reindex(df.index, method="ffill")
        iwf_safe = iwf.replace(0, np.nan)
        iwd_safe = iwd.replace(0, np.nan)
        iwf_ret = np.log(iwf_safe / iwf_safe.shift(20)).fillna(0)
        iwd_ret = np.log(iwd_safe / iwd_safe.shift(20)).fillna(0)
        feat["value_growth_spread_20d"] = iwf_ret - iwd_ret  # positive = growth > value
    else:
        feat["value_growth_spread_20d"] = 0.0

    # ── Feature 153: Risk Appetite (XLY vs XLP) ──
    if macro_df is not None and "XLY" in macro_df.columns and "XLP" in macro_df.columns:
        xly = macro_df["XLY"].reindex(df.index, method="ffill")
        xlp = macro_df["XLP"].reindex(df.index, method="ffill")
        xly_safe = xly.replace(0, np.nan)
        xlp_safe = xlp.replace(0, np.nan)
        xly_ret = np.log(xly_safe / xly_safe.shift(20)).fillna(0)
        xlp_ret = np.log(xlp_safe / xlp_safe.shift(20)).fillna(0)
        feat["risk_appetite_ratio_20d"] = xly_ret - xlp_ret  # positive = risk-on
    else:
        feat["risk_appetite_ratio_20d"] = 0.0

    # Clean up
    feat = feat.replace([np.inf, -np.inf], np.nan)
    feat = feat.fillna(0)

    return feat


print(f"[OK] Feature engineering function defined (154 features)")

In [ ]:
# Cell 5: Data Download & Preparation
# Downloads 10 years of data for all 80+ stocks + macro indicators

import time

def download_macro_data(tickers: List[str], period: str = "10y") -> pd.DataFrame:
    """Download macro/sector/credit/commodity data for all tickers."""
    macro_df = pd.DataFrame()
    for ticker in tickers:
        try:
            df = yf.download(ticker, period=period, interval="1d", progress=False)
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            if len(df) > 0:
                macro_df[ticker] = df["Close"].squeeze()
                print(f"  Macro {ticker}: {len(df)} days")
        except Exception as e:
            print(f"  Macro {ticker}: FAILED ({e})")
    return macro_df




def download_fred_data(fred_series: dict, years: int = 10) -> pd.DataFrame:
    """Download FRED macro series using the fredapi library.
    
    Args:
        fred_series: dict mapping FRED series IDs to column names
        years: number of years of history to fetch
    
    Returns:
        DataFrame with FRED series as columns, indexed by date
    """
    try:
        from fredapi import Fred
    except ImportError:
        print("  [WARN] fredapi not installed — skipping FRED data")
        return pd.DataFrame()
    
    if not FRED_API_KEY:
        print("  [WARN] No FRED_API_KEY — skipping FRED data")
        return pd.DataFrame()
    
    fred = Fred(api_key=FRED_API_KEY)
    fred_df = pd.DataFrame()
    
    from datetime import datetime, timedelta
    end_date = datetime.now()
    start_date = end_date - timedelta(days=years * 365)
    
    for series_id, col_name in fred_series.items():
        try:
            data = fred.get_series(series_id, observation_start=start_date, observation_end=end_date)
            if len(data) > 0:
                # Use uppercase column name with FRED_ prefix for macro_df
                fred_col = f"FRED_{col_name.replace('fred_', '').upper()}"
                fred_df[fred_col] = data
                print(f"  FRED {series_id} ({fred_col}): {len(data)} observations")
        except Exception as e:
            print(f"  FRED {series_id}: FAILED ({e})")
    
    return fred_df


def download_stock_data(
    symbols: List[str],
    macro_tickers: List[str],
    lookback: int,
    horizon: int,
    data_years: int = 10,
) -> Tuple[Dict[str, Tuple[np.ndarray, np.ndarray]], List[str], dict, pd.DataFrame]:
    """
    Download data for all stocks, compute features with macro indicators,
    create per-stock training samples using walk-forward approach.

    Returns:
        stock_data: dict mapping symbol -> (X, y) arrays
        feature_names: list of feature column names
        normalization_stats: per-stock mean/std for inference
        macro_df: macro data for reuse
    """
    # Download macro data first
    print("  Downloading macro indicators...")
    macro_df = download_macro_data(macro_tickers, period=f"{data_years}y")
    print(f"  Macro data: {len(macro_df)} days, {len(macro_df.columns)} indicators")

    # Download FRED data and merge into macro_df
    print("  Downloading FRED macro indicators...")
    fred_df = download_fred_data(FRED_SERIES, data_years)
    if len(fred_df) > 0:
        # Forward-fill FRED data (weekly/monthly → daily) and merge
        fred_df = fred_df.resample("D").ffill()
        macro_df = macro_df.join(fred_df, how="left")
        macro_df = macro_df.ffill()
        print(f"  FRED data merged: {len(fred_df.columns)} series\n")
    else:
        print(f"  No FRED data available\n")


    stock_data = {}
    feature_names = None
    normalization_stats = {}
    failed = []

    for i, sym in enumerate(symbols):
        print(f"  [{i+1}/{len(symbols)}] {sym}...", end=" ", flush=True)
        try:
            df = yf.download(sym, period=f"{data_years}y", interval="1d", progress=False)
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            if len(df) < lookback + horizon + 252:
                print(f"skipped (only {len(df)} days)")
                failed.append(sym)
                continue

            # Build per-stock macro_df with SECTOR_ETF, INDUSTRY_COMMODITY, and STOCK_DRIVER columns
            stock_macro = macro_df.copy()
            sector_etf = SECTOR_ETF_MAP.get(sym)
            if sector_etf and sector_etf in macro_df.columns:
                stock_macro["SECTOR_ETF"] = macro_df[sector_etf]
            elif "SPY" in macro_df.columns:
                stock_macro["SECTOR_ETF"] = macro_df["SPY"]  # fallback
            industry_commodity = INDUSTRY_COMMODITY_MAP.get(sym)
            if industry_commodity and industry_commodity in macro_df.columns:
                stock_macro["INDUSTRY_COMMODITY"] = macro_df[industry_commodity]
            # Stock-specific driver assets (v8.0)
            drivers = STOCK_SPECIFIC_DRIVERS.get(sym)
            if drivers:
                driver1_sym, driver2_sym = drivers
                if driver1_sym in macro_df.columns:
                    stock_macro["STOCK_DRIVER_1"] = macro_df[driver1_sym]
                if driver2_sym in macro_df.columns:
                    stock_macro["STOCK_DRIVER_2"] = macro_df[driver2_sym]
            features = compute_features(df, stock_macro)
            closes = df["Close"].values.flatten()

            if feature_names is None:
                feature_names = list(features.columns)
                print(f"{len(feature_names)} features, {len(df)} days")
            else:
                print(f"{len(df)} days")

            # Z-score normalize per stock
            feat_values = features.values
            feat_mean = np.nanmean(feat_values, axis=0, keepdims=True)
            feat_std = np.nanstd(feat_values, axis=0, keepdims=True) + 1e-10
            feat_norm = (feat_values - feat_mean) / feat_std
            feat_norm = np.clip(feat_norm, -5, 5)

            normalization_stats[sym] = {
                "mean": feat_mean.flatten().tolist(),
                "std": feat_std.flatten().tolist(),
            }

            # Create sliding window samples
            X_samples = []
            y_samples = []
            for j in range(lookback, len(feat_norm) - horizon):
                X_sample = feat_norm[j - lookback:j]
                current_price = closes[j]
                future_prices = closes[j + 1:j + horizon + 1]
                if len(future_prices) == horizon and current_price > 0:
                    y_sample = (future_prices - current_price) / current_price
                    X_samples.append(X_sample)
                    y_samples.append(y_sample)

            if len(X_samples) >= MIN_SAMPLES:
                stock_data[sym] = (
                    np.array(X_samples, dtype=np.float32),
                    np.array(y_samples, dtype=np.float32),
                )
            else:
                print(f"  {sym}: only {len(X_samples)} samples, skipping (min={MIN_SAMPLES})")
                failed.append(sym)

            # Rate limiting for Yahoo Finance
            if (i + 1) % 5 == 0:
                time.sleep(1)

        except Exception as e:
            print(f"FAILED ({e})")
            failed.append(sym)
            continue

    print(f"\n[DATA] {len(stock_data)} stocks ready for training")
    total_samples = sum(x.shape[0] for x, _ in stock_data.values())
    print(f"[DATA] {total_samples:,} total samples across all stocks")
    if feature_names:
        print(f"[DATA] {len(feature_names)} features per sample")
    if failed:
        print(f"[DATA] Failed/skipped: {', '.join(failed)}")

    return stock_data, feature_names or [], normalization_stats, macro_df


print("[1/5] Downloading market data and computing features...")
print(f"  {len(SCREENER_SYMBOLS)} stocks × {DATA_YEARS} years + {len(ALL_DATA_TICKERS)} data tickers\n")
stock_data, feature_names, norm_stats, macro_df = download_stock_data(
    SCREENER_SYMBOLS, ALL_DATA_TICKERS, LOOKBACK_WINDOW, FORECAST_HORIZON, DATA_YEARS
)
print(f"\n[DATA] Features ({len(feature_names)}): {feature_names[:10]}...")
print(f"[DATA] Per-stock sample counts:")
for sym in sorted(stock_data.keys(), key=lambda s: stock_data[s][0].shape[0], reverse=True)[:10]:
    print(f"  {sym}: {stock_data[sym][0].shape[0]:,} samples")
if len(stock_data) > 10:
    print(f"  ... and {len(stock_data)-10} more stocks")

In [ ]:
# Cell 6: iTransformer Model Architecture
# RevIN buffers are registered properly for clean ONNX export

import torch
import torch.nn as nn
import math


class RevIN(nn.Module):
    """
    Reversible Instance Normalization (Kim et al., ICLR 2022).
    Uses register_buffer for _mean and _std so ONNX export works cleanly.
    """
    def __init__(self, num_features: int, eps: float = 1e-5, affine: bool = True):
        super().__init__()
        self.eps = eps
        self.affine = affine
        if affine:
            self.affine_weight = nn.Parameter(torch.ones(num_features))
            self.affine_bias = nn.Parameter(torch.zeros(num_features))
        # Register as buffers so torch.export / ONNX doesn't complain
        self.register_buffer("_mean", torch.zeros(1, 1, num_features))
        self.register_buffer("_std", torch.ones(1, 1, num_features))

    def forward(self, x: torch.Tensor, mode: str = "norm") -> torch.Tensor:
        if mode == "norm":
            self._mean = x.mean(dim=1, keepdim=True).detach()
            self._std = (x.std(dim=1, keepdim=True) + self.eps).detach()
            x = (x - self._mean) / self._std
            if self.affine:
                x = x * self.affine_weight + self.affine_bias
            return x
        else:
            if self.affine:
                x = (x - self.affine_bias) / (self.affine_weight + self.eps)
            return x


class iTransformer(nn.Module):
    """
    Inverted Transformer for Time-Series Forecasting (ICLR 2024, Liu et al.).

    1. RevIN normalization (per-window)
    2. INVERT: transpose to (batch, num_variates, lookback)
    3. Shared embedding: Linear(lookback -> d_model) per variate
    4. Learnable variate tokens
    5. Transformer encoder: self-attention across variates
    6. Shared projection: Linear(d_model -> forecast_horizon)
    7. Weighted aggregation across variates
    """

    def __init__(
        self,
        num_variates: int,
        lookback: int,
        forecast_horizon: int,
        d_model: int = 128,
        n_heads: int = 8,
        n_layers: int = 3,
        d_ff: int = 256,
        dropout: float = 0.15,
        use_norm: bool = True,
    ):
        super().__init__()
        self.num_variates = num_variates
        self.lookback = lookback
        self.forecast_horizon = forecast_horizon
        self.d_model = d_model
        self.use_norm = use_norm

        if use_norm:
            self.revin = RevIN(num_variates)

        self.variate_embedding = nn.Linear(lookback, d_model)
        self.variate_tokens = nn.Parameter(torch.randn(1, num_variates, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        # enable_nested_tensor=False suppresses the warning:
        # "enable_nested_tensor is True, but self.use_nested_tensor is False
        # because encoder_layer.norm_first was True"
        self.encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=n_layers, enable_nested_tensor=False
        )
        self.projection = nn.Linear(d_model, forecast_horizon, bias=True)
        self.agg_weights = nn.Parameter(torch.ones(num_variates) / num_variates)

        self.output_head = nn.Sequential(
            nn.LayerNorm(forecast_horizon),
            nn.Linear(forecast_horizon, forecast_horizon),
            nn.Tanh(),
        )
        self._init_weights()

    def _init_weights(self):
        for name, p in self.named_parameters():
            if p.dim() > 1 and 'revin' not in name:
                nn.init.xavier_uniform_(p)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.use_norm:
            x = self.revin(x, mode="norm")
        x = x.transpose(1, 2)
        tokens = self.variate_embedding(x)
        tokens = tokens + self.variate_tokens
        encoded = self.encoder(tokens)
        per_variate_forecast = self.projection(encoded)
        weights = torch.softmax(self.agg_weights, dim=0)
        forecast = torch.einsum('bvh,v->bh', per_variate_forecast, weights)
        forecast = self.output_head(forecast)
        return forecast

    def get_variate_importance(self) -> np.ndarray:
        with torch.no_grad():
            return torch.softmax(self.agg_weights, dim=0).cpu().numpy()


print(f"[OK] iTransformer model class defined")
print(f"  Architecture: d_model={D_MODEL}, layers={N_LAYERS}, heads={N_HEADS}, dropout={DROPOUT}")

In [ ]:
# Cell 7: Per-Stock Training & ONNX Export
# Trains one iTransformer per stock, exports each to ONNX
# Uses legacy TorchScript exporter (dynamo=False) — the dynamo exporter cannot handle
# RevIN's dynamic buffer reassignment or the string mode parameter in forward()

from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import gc
import json
import onnx
import onnxruntime as ort
import warnings

print("[2/5] Training per-stock models...")
print(f"  {len(stock_data)} stocks to train\n")

os.makedirs("per_stock", exist_ok=True)

per_stock_metrics = {}
per_stock_failed = []
all_histories = {}  # For visualization

for idx, (sym, (X_stock, y_stock)) in enumerate(stock_data.items()):
    n_samples = X_stock.shape[0]
    print(f"  [{idx+1}/{len(stock_data)}] {sym} ({n_samples:,} samples)...", end=" ", flush=True)

    try:
        # Walk-forward split (70/15/15 chronological)
        n = X_stock.shape[0]
        train_end = int(n * TRAIN_SPLIT)
        val_end = int(n * VAL_SPLIT)

        X_train = X_stock[:train_end]
        y_train = y_stock[:train_end]
        X_val = X_stock[train_end:val_end]
        y_val = y_stock[train_end:val_end]
        X_test = X_stock[val_end:]
        y_test = y_stock[val_end:]

        if len(X_val) < 10 or len(X_test) < 10:
            print("skipped (insufficient val/test data)")
            per_stock_failed.append(sym)
            continue

        # Create per-stock model
        stock_model = iTransformer(
            num_variates=X_stock.shape[2],
            lookback=X_stock.shape[1],
            forecast_horizon=y_stock.shape[1],
            d_model=D_MODEL,
            n_layers=N_LAYERS,
            n_heads=N_HEADS,
            d_ff=D_FF,
            dropout=DROPOUT,
            use_norm=True,
        ).to(device)

        train_ds = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train))
        val_ds = TensorDataset(torch.FloatTensor(X_val), torch.FloatTensor(y_val))
        test_ds = TensorDataset(torch.FloatTensor(X_test), torch.FloatTensor(y_test))

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, pin_memory=True)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, pin_memory=True)

        optimizer = torch.optim.AdamW(
            stock_model.parameters(), lr=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY, betas=(0.9, 0.999),
        )

        def lr_lambda(epoch):
            if epoch < WARMUP_EPOCHS:
                return (epoch + 1) / WARMUP_EPOCHS
            progress = (epoch - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)
            return 0.5 * (1 + math.cos(math.pi * progress))

        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
        criterion = nn.HuberLoss(delta=0.02)

        best_val_loss = float("inf")
        best_state = None
        patience_counter = 0
        history = {"train_loss": [], "val_loss": [], "val_dir_acc": []}

        for epoch in range(EPOCHS):
            stock_model.train()
            train_loss = 0
            for X_batch, y_batch in train_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                optimizer.zero_grad()
                pred = stock_model(X_batch)
                loss = criterion(pred, y_batch)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(stock_model.parameters(), 1.0)
                optimizer.step()
                train_loss += loss.item()
            train_loss /= len(train_loader)

            stock_model.eval()
            val_loss = 0
            val_preds, val_trues = [], []
            with torch.no_grad():
                for X_batch, y_batch in val_loader:
                    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                    pred = stock_model(X_batch)
                    val_loss += criterion(pred, y_batch).item()
                    val_preds.append(pred.cpu())
                    val_trues.append(y_batch.cpu())
            val_loss /= len(val_loader)

            vp = torch.cat(val_preds)
            vt = torch.cat(val_trues)
            dir_acc = ((vp[:, -1] > 0) == (vt[:, -1] > 0)).float().mean().item() * 100

            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["val_dir_acc"].append(dir_acc)

            scheduler.step()

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {k: v.cpu().clone() for k, v in stock_model.state_dict().items()}
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= PATIENCE:
                break

        # Load best weights
        if best_state:
            stock_model.load_state_dict(best_state)
        stock_model = stock_model.to(device)

        # Test evaluation
        stock_model.eval()
        test_preds, test_trues = [], []
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                pred = stock_model(X_batch.to(device))
                test_preds.append(pred.cpu())
                test_trues.append(y_batch)

        test_preds_t = torch.cat(test_preds)
        test_trues_t = torch.cat(test_trues)

        stock_mse = ((test_preds_t - test_trues_t) ** 2).mean().item()
        stock_mae = (test_preds_t - test_trues_t).abs().mean().item()

        # Per-horizon directional accuracy
        horizon_dir_acc = []
        for h in range(test_preds_t.shape[1]):
            acc = ((test_preds_t[:, h] > 0) == (test_trues_t[:, h] > 0)).float().mean().item() * 100
            horizon_dir_acc.append(acc)

        dir_7d = horizon_dir_acc[6] if len(horizon_dir_acc) > 6 else horizon_dir_acc[0]
        dir_14d = horizon_dir_acc[13] if len(horizon_dir_acc) > 13 else horizon_dir_acc[0]
        dir_30d = horizon_dir_acc[29] if len(horizon_dir_acc) > 29 else horizon_dir_acc[-1]
        dir_60d = horizon_dir_acc[-1]

        # ── ONNX Export ──
        # Use legacy TorchScript exporter (dynamo=False) because the dynamo exporter
        # fails on RevIN's dynamic buffer reassignment and string mode parameter.
        # The dynamic_axes deprecation warning is harmless — the export works correctly.
        stock_model_cpu = stock_model.cpu()
        stock_model_cpu.eval()
        dummy = torch.randn(1, LOOKBACK_WINDOW, len(feature_names))

        onnx_path = f"per_stock/{sym}.onnx"
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message=".*dynamic_axes.*dynamo.*")
            torch.onnx.export(
                stock_model_cpu, dummy, onnx_path,
                input_names=["features"], output_names=["forecast"],
                dynamic_axes={"features": {0: "batch_size"}, "forecast": {0: "batch_size"}},
                opset_version=18,
                do_constant_folding=True,
                dynamo=False,
            )

        # Consolidate external data into single file
        onnx_m = onnx.load(onnx_path, load_external_data=True)
        onnx.save_model(onnx_m, onnx_path, save_as_external_data=False)
        data_path = f"{onnx_path}.data"
        if os.path.exists(data_path):
            os.remove(data_path)

        # Verify with ONNX Runtime
        session = ort.InferenceSession(onnx_path)
        test_input = np.random.randn(1, LOOKBACK_WINDOW, len(feature_names)).astype(np.float32)
        onnx_out = session.run(None, {"features": test_input})[0]
        with torch.no_grad():
            torch_out = stock_model_cpu(torch.FloatTensor(test_input)).numpy()
        max_diff = np.abs(torch_out - onnx_out).max()
        del session

        onnx_size_s = os.path.getsize(onnx_path) / (1024 * 1024)

        # Save state dict for feature importance analysis (Cell 9)
        torch.save(stock_model.state_dict(), f"per_stock/{sym}_state.pt")

        per_stock_metrics[sym] = {
            "mse": stock_mse, "mae": stock_mae,
            "dir_acc_7d": dir_7d, "dir_acc_14d": dir_14d,
            "dir_acc_30d": dir_30d, "dir_acc_60d": dir_60d,
            "horizon_dir_acc": horizon_dir_acc,
            "num_samples": n_samples,
            "epochs_trained": epoch + 1,
            "best_val_loss": best_val_loss,
            "onnx_size_mb": round(onnx_size_s, 2),
            "onnx_torch_diff": float(max_diff),
        }

        all_histories[sym] = history

        print(f"dir_30d={dir_30d:.1f}%, dir_60d={dir_60d:.1f}%, {epoch+1} epochs, {onnx_size_s:.1f}MB, onnx_diff={max_diff:.2e}")

        # Clean up GPU memory
        del stock_model, stock_model_cpu, optimizer, scheduler
        del train_ds, val_ds, test_ds, train_loader, val_loader, test_loader
        del test_preds, test_trues, test_preds_t, test_trues_t
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    except Exception as e:
        print(f"FAILED ({e})")
        per_stock_failed.append(sym)
        gc.collect()
        continue

print(f"\n[TRAINING] Trained {len(per_stock_metrics)} models, {len(per_stock_failed)} failed/skipped")
if per_stock_metrics:
    avg_dir30 = np.mean([m["dir_acc_30d"] for m in per_stock_metrics.values()])
    avg_dir7 = np.mean([m["dir_acc_7d"] for m in per_stock_metrics.values()])
    avg_dir60 = np.mean([m["dir_acc_60d"] for m in per_stock_metrics.values()])
    print(f"[TRAINING] Average 7d directional accuracy:  {avg_dir7:.1f}%")
    print(f"[TRAINING] Average 30d directional accuracy: {avg_dir30:.1f}%")    print(f"[TRAINING] Average 60d directional accuracy: {avg_dir60:.1f}%")


In [ ]:
# Cell 8: Training Visualization

import matplotlib.pyplot as plt

# Summary bar chart of 30d directional accuracy per stock
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("PutStrike iTransformer — Per-Stock Training Results", fontsize=14, fontweight="bold")

# Top-left: 30d directional accuracy per stock
ax1 = axes[0, 0]
symbols_sorted = sorted(per_stock_metrics.keys(), key=lambda s: per_stock_metrics[s]["dir_acc_30d"], reverse=True)
accs = [per_stock_metrics[s]["dir_acc_30d"] for s in symbols_sorted]
colors = ["green" if a > 55 else "steelblue" if a > 50 else "orange" for a in accs]
ax1.barh(symbols_sorted[:30], accs[:30], color=colors[:30], alpha=0.8)
ax1.axvline(x=50, color="red", linestyle="--", alpha=0.5, label="Random (50%)")
ax1.set_xlabel("30d Directional Accuracy (%)")
ax1.set_title(f"Top 30 Stocks — 30d Dir Accuracy")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.invert_yaxis()

# Top-right: Distribution of directional accuracies
ax2 = axes[0, 1]
all_dir30 = [m["dir_acc_30d"] for m in per_stock_metrics.values()]
all_dir7 = [m["dir_acc_7d"] for m in per_stock_metrics.values()]
all_dir60 = [m["dir_acc_60d"] for m in per_stock_metrics.values()]
ax2.hist(all_dir30, bins=20, alpha=0.6, label="30d", color="steelblue")
ax2.hist(all_dir7, bins=20, alpha=0.6, label="7d", color="coral")
ax2.hist(all_dir60, bins=20, alpha=0.6, label="60d", color="mediumpurple")
ax2.axvline(x=50, color="red", linestyle="--", alpha=0.5, label="Random")
ax2.set_xlabel("Directional Accuracy (%)")
ax2.set_ylabel("Count")
ax2.set_title("Distribution of Dir Accuracy Across Stocks")
ax2.legend()
ax2.grid(True, alpha=0.3)

# Bottom-left: Average per-horizon accuracy across all stocks
ax3 = axes[1, 0]
avg_horizon = np.mean([m["horizon_dir_acc"] for m in per_stock_metrics.values()], axis=0)
days = list(range(1, len(avg_horizon) + 1))
ax3.bar(days, avg_horizon, alpha=0.7, color="steelblue")
ax3.axhline(y=50, color="red", linestyle="--", alpha=0.5, label="Random")
for d, label in [(7, "7d"), (14, "14d"), (30, "30d"), (45, "45d"), (60, "60d")]:
    if d <= len(days):
        ax3.axvline(x=d, color="orange", linestyle=":", alpha=0.5)
        ax3.text(d, max(avg_horizon) + 0.5, label, ha="center", fontsize=7, color="orange")
ax3.set_xlabel("Forecast Day")
ax3.set_ylabel("Average Dir Accuracy (%)")
ax3.set_title("Average Per-Horizon Accuracy (All Stocks)")
ax3.legend()
ax3.grid(True, alpha=0.3)

# Bottom-right: Training epochs distribution
ax4 = axes[1, 1]
epochs_list = [m["epochs_trained"] for m in per_stock_metrics.values()]
ax4.hist(epochs_list, bins=20, alpha=0.7, color="coral")
ax4.set_xlabel("Epochs Trained")
ax4.set_ylabel("Count")
ax4.set_title(f"Training Duration (max={EPOCHS}, patience={PATIENCE})")
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_results.png", dpi=150, bbox_inches="tight")
plt.show()

# Print summary stats
print(f"\n[SUMMARY]")
print(f"  Models trained:     {len(per_stock_metrics)}")
print(f"  Avg 7d Dir Acc:     {np.mean(all_dir7):.1f}% (std={np.std(all_dir7):.1f}%)")
print(f"  Avg 30d Dir Acc:    {np.mean(all_dir30):.1f}% (std={np.std(all_dir30):.1f}%)")
print(f"  Avg 60d Dir Acc:    {np.mean(all_dir60):.1f}% (std={np.std(all_dir60):.1f}%)")
print(f"  Avg epochs:         {np.mean(epochs_list):.0f}")
print(f"  Best stock (30d):   {symbols_sorted[0]} ({per_stock_metrics[symbols_sorted[0]]['dir_acc_30d']:.1f}%)")
print(f"  Worst stock (30d):  {symbols_sorted[-1]} ({per_stock_metrics[symbols_sorted[-1]]['dir_acc_30d']:.1f}%)")symbols_sorted_60 = sorted(per_stock_metrics.keys(), key=lambda s: per_stock_metrics[s]["dir_acc_60d"], reverse=True)
print(f"  Best stock (60d):   {symbols_sorted_60[0]} ({per_stock_metrics[symbols_sorted_60[0]]['dir_acc_60d']:.1f}%)")
print(f"  Worst stock (60d):  {symbols_sorted_60[-1]} ({per_stock_metrics[symbols_sorted_60[-1]]['dir_acc_60d']:.1f}%)")


In [ ]:
# Cell 9: Feature Importance Analysis
# Computes permutation-based feature importance for each per-stock model.
# Produces a report that can be fed back to Claude for analysis.

import pandas as pd
from collections import defaultdict

print("[FEATURE IMPORTANCE] Computing permutation importance per stock...")
print("  This shuffles each feature column and measures prediction degradation.\n")

def compute_permutation_importance(
    model, X_test, y_test, feature_names, n_repeats=5, device="cpu"
):
    """
    Compute permutation importance for each feature.
    
    For each feature:
    1. Record baseline MSE on test set
    2. Shuffle that feature column n_repeats times
    3. Measure mean MSE increase (importance = how much prediction degrades)
    
    Returns dict: feature_name -> {importance, std, rank}
    """
    model.eval()
    X_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_tensor = torch.tensor(y_test, dtype=torch.float32).to(device)
    
    with torch.no_grad():
        baseline_pred = model(X_tensor)
        baseline_mse = ((baseline_pred - y_tensor) ** 2).mean().item()
    
    importances = {}
    num_features = X_test.shape[2]
    
    for feat_idx in range(num_features):
        mse_increases = []
        for _ in range(n_repeats):
            X_shuffled = X_test.copy()
            # Shuffle this feature across all samples (break the temporal pattern)
            perm = np.random.permutation(X_shuffled.shape[0])
            X_shuffled[:, :, feat_idx] = X_shuffled[perm, :, feat_idx]
            
            X_shuf_tensor = torch.tensor(X_shuffled, dtype=torch.float32).to(device)
            with torch.no_grad():
                shuf_pred = model(X_shuf_tensor)
                shuf_mse = ((shuf_pred - y_tensor) ** 2).mean().item()
            
            mse_increases.append(shuf_mse - baseline_mse)
        
        feat_name = feature_names[feat_idx] if feat_idx < len(feature_names) else f"feature_{feat_idx}"
        importances[feat_name] = {
            "importance": float(np.mean(mse_increases)),
            "std": float(np.std(mse_increases)),
        }
    
    # Add ranks
    sorted_feats = sorted(importances.keys(), key=lambda f: importances[f]["importance"], reverse=True)
    for rank, feat in enumerate(sorted_feats, 1):
        importances[feat]["rank"] = rank
    
    return importances, baseline_mse


# Compute importance for each trained stock
all_importances = {}
feature_importance_summary = defaultdict(list)

for sym in sorted(per_stock_metrics.keys()):
    if sym not in stock_data:
        continue
    
    X, y = stock_data[sym]
    n_total = X.shape[0]
    n_train = int(n_total * TRAIN_SPLIT)
    n_val = int(n_total * VAL_SPLIT)
    X_test = X[n_val:]
    y_test = y[n_val:]
    
    if len(X_test) < 50:
        print(f"  {sym}: skipped (only {len(X_test)} test samples)")
        continue
    
    # Load the trained model
    onnx_path = f"per_stock/{sym}.onnx"
    if not os.path.exists(onnx_path):
        continue
    
    # Re-create model and load weights from the training state
    # (We use the model that was saved during training)
    try:
        model = iTransformer(
            num_variates=X.shape[2], lookback=LOOKBACK_WINDOW,
            forecast_horizon=FORECAST_HORIZON, d_model=D_MODEL,
            n_layers=N_LAYERS, n_heads=N_HEADS, d_ff=D_FF, dropout=DROPOUT,
        ).to(device)
        
        # Load saved state dict if available
        state_path = f"per_stock/{sym}_state.pt"
        if os.path.exists(state_path):
            model.load_state_dict(torch.load(state_path, map_location=device))
        else:
            # If no state dict, skip (can't compute importance without trained weights)
            print(f"  {sym}: no state dict found, skipping importance")
            continue
        
        importances, baseline = compute_permutation_importance(
            model, X_test, y_test, feature_names, n_repeats=3, device=device
        )
        all_importances[sym] = {
            "importances": importances,
            "baseline_mse": baseline,
            "test_samples": len(X_test),
        }
        
        # Collect per-feature importance across stocks
        for feat, vals in importances.items():
            feature_importance_summary[feat].append(vals["importance"])
        
        # Print top 10 for this stock
        top10 = sorted(importances.items(), key=lambda x: x[1]["importance"], reverse=True)[:10]
        print(f"  {sym} (baseline MSE: {baseline:.6f}, {len(X_test)} test samples):")
        for feat, vals in top10:
            print(f"    #{vals['rank']:3d} {feat:40s} importance={vals['importance']:.6f} ±{vals['std']:.6f}")
        
    except Exception as e:
        print(f"  {sym}: FAILED ({e})")
        continue

# ── Aggregate Analysis ──
print("\n" + "=" * 80)
print("AGGREGATE FEATURE IMPORTANCE (across all stocks)")
print("=" * 80)

# Compute mean importance per feature across all stocks
agg_importance = {}
for feat, imp_list in feature_importance_summary.items():
    agg_importance[feat] = {
        "mean_importance": float(np.mean(imp_list)),
        "std_importance": float(np.std(imp_list)),
        "num_stocks": len(imp_list),
        "max_importance": float(np.max(imp_list)),
        "min_importance": float(np.min(imp_list)),
    }

# Sort by mean importance
sorted_agg = sorted(agg_importance.items(), key=lambda x: x[1]["mean_importance"], reverse=True)

# Feature categories for analysis
FEATURE_CATEGORIES = {
    "Price Action (0-9)": list(range(10)),
    "Momentum (10-33)": list(range(10, 34)),
    "Returns & Volatility (34-46)": list(range(34, 47)),
    "Statistical (47-62)": list(range(47, 63)),
    "Calendar & Trend (63-72)": list(range(63, 73)),
    "Macro (73-82)": list(range(73, 83)),
    "Relative Strength (83-89)": list(range(83, 90)),
    "Advanced Volume (90-93)": list(range(90, 94)),
    "Price Structure (94-98)": list(range(94, 99)),
    "Regime Detection (99-103)": list(range(99, 104)),
    "Intermarket (104-107)": list(range(104, 108)),
    "Sector/Credit (108-119)": list(range(108, 120)),
    "FRED Macro (120-125)": list(range(120, 126)),
    "Gamma Squeeze Proxies (126-131)": list(range(126, 132)),
    "Market Breadth (132-135)": list(range(132, 136)),
    "Sentiment Proxies (136-139)": list(range(136, 140)),
    "Stock-Specific Drivers (140-143)": list(range(140, 144)),
    "FRED Extended (144-145)": list(range(144, 146)),
    "FRED Rates (146-147)": list(range(146, 148)),
    "FRED FX (148-149)": list(range(148, 150)),
    "Tail Risk (150-151)": list(range(150, 152)),
    "Style Rotation (152-153)": list(range(152, 154)),
}

print("\n── Top 30 Most Important Features (averaged across stocks) ──")
for rank, (feat, vals) in enumerate(sorted_agg[:30], 1):
    print(f"  #{rank:3d} {feat:40s} mean={vals['mean_importance']:.6f} ±{vals['std_importance']:.6f} (max={vals['max_importance']:.6f}, {vals['num_stocks']} stocks)")

print("\n── Bottom 10 Least Important Features ──")
for rank, (feat, vals) in enumerate(sorted_agg[-10:], len(sorted_agg) - 9):
    print(f"  #{rank:3d} {feat:40s} mean={vals['mean_importance']:.6f}")

# Category-level analysis
print("\n── Category-Level Importance (mean of features in category) ──")
cat_importance = {}
for cat_name, indices in FEATURE_CATEGORIES.items():
    cat_feats = [feature_names[i] for i in indices if i < len(feature_names)]
    cat_imps = [agg_importance[f]["mean_importance"] for f in cat_feats if f in agg_importance]
    if cat_imps:
        cat_importance[cat_name] = {
            "mean": float(np.mean(cat_imps)),
            "max": float(np.max(cat_imps)),
            "num_features": len(cat_imps),
        }

for cat, vals in sorted(cat_importance.items(), key=lambda x: x[1]["mean"], reverse=True):
    print(f"  {cat:40s} mean={vals['mean']:.6f} max={vals['max']:.6f} ({vals['num_features']} features)")

# ── Per-Stock Top Features (for Claude analysis) ──
print("\n── Per-Stock Top 5 Features ──")
for sym in sorted(all_importances.keys()):
    data = all_importances[sym]
    top5 = sorted(data["importances"].items(), key=lambda x: x[1]["importance"], reverse=True)[:5]
    features_str = ", ".join([f"{f}({v['importance']:.4f})" for f, v in top5])
    print(f"  {sym}: {features_str}")

# ── New Features Impact Analysis ──
print("\n── NEW v8.0 Feature Categories Impact ──")
new_categories = {
    "Gamma Squeeze Proxies (126-131)": list(range(126, 132)),
    "Market Breadth (132-135)": list(range(132, 136)),
    "Sentiment Proxies (136-139)": list(range(136, 140)),
    "Stock-Specific Drivers (140-143)": list(range(140, 144)),
    "FRED Extended (144-145)": list(range(144, 146)),
    "FRED Rates (146-147)": list(range(146, 148)),
    "FRED FX (148-149)": list(range(148, 150)),
    "Tail Risk (150-151)": list(range(150, 152)),
    "Style Rotation (152-153)": list(range(152, 154)),
}
for cat_name, indices in new_categories.items():
    cat_feats = [feature_names[i] for i in indices if i < len(feature_names)]
    cat_imps = [agg_importance[f]["mean_importance"] for f in cat_feats if f in agg_importance]
    if cat_imps:
        print(f"  {cat_name}:")
        for f in cat_feats:
            if f in agg_importance:
                v = agg_importance[f]
                print(f"    {f:40s} mean={v['mean_importance']:.6f} (rank among {len(sorted_agg)} features)")
        print(f"    Category avg: {np.mean(cat_imps):.6f}")
    else:
        print(f"  {cat_name}: No data (features not in trained models)")

# ── Save importance report as JSON for Claude analysis ──
importance_report = {
    "summary": "Feature importance analysis for PutStrike iTransformer v8.0",
    "method": "Permutation importance (3 repeats per feature per stock)",
    "num_stocks_analyzed": len(all_importances),
    "num_features": len(feature_names),
    "aggregate_importance": {f: v for f, v in sorted_agg},
    "category_importance": cat_importance,
    "per_stock": {
        sym: {
            "baseline_mse": data["baseline_mse"],
            "test_samples": data["test_samples"],
            "top_10": [
                {"feature": f, **v}
                for f, v in sorted(data["importances"].items(),
                                   key=lambda x: x[1]["importance"], reverse=True)[:10]
            ],
        }
        for sym, data in all_importances.items()
    },
}

with open("feature_importance_report.json", "w") as f:
    json.dump(importance_report, f, indent=2)
print(f"\n  Saved: feature_importance_report.json (feed this to Claude for analysis)")

print(f"\n[FEATURE IMPORTANCE] Done. Analyzed {len(all_importances)} stocks.")

In [ ]:
# Cell 9: Save Config & Push to HuggingFace Hub

from huggingface_hub import HfApi, create_repo
import glob as glob_module

print("[3/5] Saving configs and pushing to HuggingFace Hub...")

# Save per-stock config
per_stock_config = {
    "model_name": "putstrike-itransformer-per-stock",
    "version": "9.0",
    "feature_names": feature_names,
    "num_features": len(feature_names),
    "lookback": LOOKBACK_WINDOW,
    "forecast_horizon": FORECAST_HORIZON,
    "architecture": {
        "type": "iTransformer",
        "d_model": D_MODEL,
        "n_layers": N_LAYERS,
        "n_heads": N_HEADS,
        "d_ff": D_FF,
        "dropout": DROPOUT,
        "use_revin": True,
    },
    "training": {
        "symbols": list(per_stock_metrics.keys()),
        "num_stocks": len(per_stock_metrics),
        "epochs": EPOCHS,
        "patience": PATIENCE,
        "batch_size": BATCH_SIZE,
        "lr": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "min_samples": MIN_SAMPLES,
        "data_years": DATA_YEARS,
        "loss_function": "HuberLoss(delta=0.02)",
        "optimizer": "AdamW",
        "lr_schedule": "warmup + cosine decay",
        "validation": "walk-forward (70/15/15)",
    },
    "per_stock_metrics": per_stock_metrics,
    "failed_stocks": per_stock_failed,
    "normalization_stats": norm_stats,
}

with open("per_stock/per_stock_config.json", "w") as f:
    json.dump(per_stock_config, f, indent=2)

# Also save as model_config.json (website expects this)
model_config = {
    "model_name": "putstrike-itransformer-per-stock",
    "version": "9.0",
    "feature_names": feature_names,
    "num_features": len(feature_names),
    "lookback": LOOKBACK_WINDOW,
    "forecast_horizon": FORECAST_HORIZON,
    "architecture": {
        "type": "iTransformer (per-stock)",
        "d_model": D_MODEL,
        "n_layers": N_LAYERS,
        "n_heads": N_HEADS,
        "d_ff": D_FF,
        "dropout": DROPOUT,
        "use_revin": True,
        "parameters": None,  # set below
    },
    "training": {
        "symbols": list(per_stock_metrics.keys()),
        "num_stocks": len(per_stock_metrics),
        "total_samples": sum(m["num_samples"] for m in per_stock_metrics.values()),
        "epochs_trained": int(np.mean([m["epochs_trained"] for m in per_stock_metrics.values()])),
        "best_val_loss": float(np.mean([m["best_val_loss"] for m in per_stock_metrics.values()])),
    },
    "test_metrics": {
        "mse": float(np.mean([m["mse"] for m in per_stock_metrics.values()])),
        "mae": float(np.mean([m["mae"] for m in per_stock_metrics.values()])),
        "dir_acc_7d": float(np.mean([m["dir_acc_7d"] for m in per_stock_metrics.values()])),
        "dir_acc_14d": float(np.mean([m["dir_acc_14d"] for m in per_stock_metrics.values()])),
        "dir_acc_30d": float(np.mean([m["dir_acc_30d"] for m in per_stock_metrics.values()])),
        "dir_acc_60d": float(np.mean([m["dir_acc_60d"] for m in per_stock_metrics.values()])),
        "horizon_dir_acc": np.mean([m["horizon_dir_acc"] for m in per_stock_metrics.values()], axis=0).tolist(),
    },
    "normalization_stats": norm_stats,
}

# Calculate param count from one model
dummy_model = iTransformer(
    num_variates=len(feature_names), lookback=LOOKBACK_WINDOW,
    forecast_horizon=FORECAST_HORIZON, d_model=D_MODEL,
    n_layers=N_LAYERS, n_heads=N_HEADS, d_ff=D_FF, dropout=DROPOUT,
)
model_config["architecture"]["parameters"] = sum(p.numel() for p in dummy_model.parameters())
del dummy_model

with open("model_config.json", "w") as f:
    json.dump(model_config, f, indent=2)

print(f"  Saved: per_stock/per_stock_config.json")
print(f"  Saved: model_config.json")

# ── Push to HuggingFace Hub ──
print(f"\n[4/5] Pushing to HuggingFace Hub...")

api = HfApi(token=HF_TOKEN)

# Create repo if it doesn't exist
try:
    create_repo(HF_REPO_ID, token=HF_TOKEN, repo_type="model", exist_ok=True)
    print(f"  Repo: https://huggingface.co/{HF_REPO_ID}")
except Exception as e:
    print(f"  Repo creation: {e}")

# Upload model_config.json (website needs this)
api.upload_file(
    path_or_fileobj="model_config.json",
    path_in_repo="model_config.json",
    repo_id=HF_REPO_ID,
    token=HF_TOKEN,
)
print(f"  Uploaded: model_config.json")

# Upload per-stock models
per_stock_onnx_files = sorted(glob_module.glob("per_stock/*.onnx"))
print(f"\n  Uploading {len(per_stock_onnx_files)} per-stock models...")
for onnx_file in per_stock_onnx_files:
    sym_name = os.path.basename(onnx_file).replace(".onnx", "")
    file_size = os.path.getsize(onnx_file) / (1024 * 1024)
    api.upload_file(
        path_or_fileobj=onnx_file,
        path_in_repo=f"per_stock/{sym_name}.onnx",
        repo_id=HF_REPO_ID,
        token=HF_TOKEN,
    )
    print(f"    {sym_name}.onnx ({file_size:.1f} MB)")

# Upload per-stock config
api.upload_file(
    path_or_fileobj="per_stock/per_stock_config.json",
    path_in_repo="per_stock/per_stock_config.json",
    repo_id=HF_REPO_ID,
    token=HF_TOKEN,
)
print(f"  Uploaded: per_stock/per_stock_config.json")

# Upload training visualization
if os.path.exists("training_results.png"):
    api.upload_file(
        path_or_fileobj="training_results.png",
        path_in_repo="training_results.png",
        repo_id=HF_REPO_ID,
        token=HF_TOKEN,
    )
    print(f"  Uploaded: training_results.png")

# Upload README
avg_dir30 = np.mean([m["dir_acc_30d"] for m in per_stock_metrics.values()])
avg_dir7 = np.mean([m["dir_acc_7d"] for m in per_stock_metrics.values()])
param_count = model_config["architecture"]["parameters"]

readme_content = f"""---
tags:
  - time-series
  - finance
  - itransformer
  - onnx
  - stock-prediction
license: mit
---

# PutStrike iTransformer — Per-Stock Forecasting Models

**iTransformer** (ICLR 2024) — individual models trained per stock for {FORECAST_HORIZON}-day price forecasting.

## Per-Stock Models

**{{len(per_stock_metrics)}} individual per-stock models**, each under `per_stock/{{SYMBOL}}.onnx`.

- **Architecture**: iTransformer with RevIN ({param_count:,} parameters each)
- **Input**: {LOOKBACK_WINDOW} days x {len(feature_names)} features (OHLCV technicals + macro + relative strength)
- **Output**: {FORECAST_HORIZON}-day forward return forecast
- **Training**: Walk-forward validation (70/15/15), HuberLoss(delta=0.02)
- **Average 7-day directional accuracy**: {avg_dir7:.1f}%
- **Average 30-day directional accuracy**: {avg_dir30:.1f}%

## v9.0 Features (154 total — includes gamma squeeze + sentiment + stock-specific drivers + tail risk + style rotation)

- 83 original: OHLCV technicals, momentum, volume, volatility, statistics, calendar, macro
- 4 relative strength: stock vs SPY returns (5/20/60d), SPY correlation
- 3 cross-asset: VIX correlation, rolling beta, volume-price correlation
- 4 advanced volume: MFI, A/D line, VWAP deviation, Force Index
- 5 price structure: range position, ATR ratio, consecutive days, candle body
- 5 statistical regime: Hurst exponent, Parkinson/GK volatility, consistency, tail ratio
- 4 intermarket: SPY momentum, gold/oil ratio, DXY-VIX interaction
- 3 sector ETF: stock vs sector ETF returns (5/20d), sector correlation
- 4 credit market: HYG/TLT returns, credit spread proxy, HYG-SPY divergence
- 1 VIX term structure: VIX9D/VIX short-term fear ratio
- 2 industry commodity: per-stock commodity correlation and return
- 2 intermarket extended: copper/gold ratio change, BTC sentiment
- 6 **gamma squeeze proxies**: volume acceleration, price-volume momentum, range expansion,
  gap acceleration, squeeze breakout signal, volume-price impact
- 4 **market breadth & rotation**: tech rotation (QQQ-SPY), small cap rotation (IWM-SPY),
  semiconductor momentum (SOX), biotech momentum (XBI)
- 4 **sentiment proxies**: realized/implied vol ratio, VIX-SPY short corr,
  credit momentum 10d, fear composite
- 4 **stock-specific drivers**: per-company primary/secondary driving asset returns & correlations
  (90 unique driver mappings across SOX, IGV, HACK, KRE, ITA, XOP, IBB, XHB, XRT, LIT, etc.)
- 2 **FRED extended**: St. Louis Fed Financial Stress Index, 10Y-3M yield spread
- 2 **FRED rates**: Federal Funds Rate level and 20d change (monetary policy stance)
- 2 **FRED FX**: JPY/USD 20d change and z-score (carry trade unwinding proxy)
- 2 **tail risk**: CBOE SKEW level and 20d z-score (options tail risk pricing)
- 1 **value/growth rotation**: IWF vs IWD 20d return spread
- 1 **risk appetite**: XLY vs XLP 20d return spread (consumer discretionary vs staples)

## Usage

```python
import onnxruntime as ort
import numpy as np

# Load per-stock model
session = ort.InferenceSession("per_stock/AAPL.onnx")

# features shape: (1, {LOOKBACK_WINDOW}, {len(feature_names)})
output = session.run(None, {{"features": features}})[0]
# output shape: (1, {FORECAST_HORIZON}) — predicted returns
```

## Disclaimer

This model is for research and educational purposes only. Not financial advice.
"""

api.upload_file(
    path_or_fileobj=readme_content.encode(),
    path_in_repo="README.md",
    repo_id=HF_REPO_ID,
    token=HF_TOKEN,
)
print(f"  Uploaded: README.md")

print(f"\n  Models available at: https://huggingface.co/{HF_REPO_ID}")
print(f"  Per-stock: https://huggingface.co/{HF_REPO_ID}/resolve/main/per_stock/{{SYMBOL}}.onnx")

In [ ]:
# Cell 10: Final Summary

print("=" * 60)
print("  PutStrike iTransformer — Training Complete!")
print("=" * 60)

avg_dir30 = np.mean([m["dir_acc_30d"] for m in per_stock_metrics.values()])
avg_dir7 = np.mean([m["dir_acc_7d"] for m in per_stock_metrics.values()])
total_size = sum(m["onnx_size_mb"] for m in per_stock_metrics.values())

print(f"""
  Architecture: Per-Stock iTransformer v4.0
  Stocks trained: {len(per_stock_metrics)}
  Features: {len(feature_names)}
  Total ONNX size: {total_size:.1f} MB ({len(per_stock_metrics)} models)

  Average Directional Accuracy:
    7-day:  {avg_dir7:.1f}%
    14-day: {np.mean([m['dir_acc_14d'] for m in per_stock_metrics.values()]):.1f}%
    30-day: {avg_dir30:.1f}%
    60-day: {np.mean([m['dir_acc_60d'] for m in per_stock_metrics.values()]):.1f}%

  Next Steps:
  1. Set HF_REPO_ID in your website's .env:
     NEXT_PUBLIC_HF_REPO_ID={HF_REPO_ID}
  2. The website will auto-download the per-stock ONNX model for each ticker
  3. Predictions run via onnxruntime-web (WASM, no GPU needed)
  4. Re-run this notebook periodically to retrain with fresh data
""")

print("[5/5] Done!")